In [ ]:
# Cell 1. 环境、路径和正式实验配置（矩阵/Numba加速版）

import os

# 重要：Numba负责VMD并行，FFT由SciPy单独控制worker。
# 在import numba之前明确允许48个Numba/OpenMP线程；
# BLAS内部仍固定单线程，避免MKL/OpenBLAS自行再开48线程。
_TARGET_THREADS_ENV = min(32, os.cpu_count() or 32)
os.environ["NUMBA_NUM_THREADS"] = str(_TARGET_THREADS_ENV)
os.environ["OMP_NUM_THREADS"] = str(_TARGET_THREADS_ENV)
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

from pathlib import Path
import gc
import json
import pickle
import math
import time
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
from netCDF4 import Dataset, num2date
from scipy.signal import hilbert
from scipy.special import gamma
from scipy import fft as sp_fft
from tqdm import tqdm

try:
    import numba
    from numba import njit, prange, set_num_threads, get_num_threads
except Exception as e:
    raise ImportError(
        "本加速Notebook需要numba。请先在当前环境安装numba，例如：conda install numba"
    ) from e

warnings.filterwarnings("ignore")

# 1. 数据目录
SOURCE_DIR = Path(os.environ.get("YANGTZE_DATA_DIR","data"))
DATA_DIR = Path(os.environ.get("YANGTZE_VMD_DIR", str(Path(os.environ.get("YANGTZE_WORK_DIR","work")) / "vmd")))
DATA_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_SPECS = {
    "SP": {
        "filename": "Surface-pressure.nc",
        "var_name": "sp",
        "output_stem": "Surface-pressure",
    },
    "T2M": {
        "filename": "2m-temperature.nc",
        "var_name": "t2m",
        "output_stem": "2m-temperature",
    },
    "D2M": {
        "filename": "2m-dewpoint-temperature.nc",
        "var_name": "d2m",
        "output_stem": "2m-dewpoint-temperature",
    },
}

# 2. 与CNN-LSTM完全一致的样本划分参数
TIMESTEP = 6
TEST_SIZE = 0.2
VALID_SIZE = 0.1

# 3. VMD参数搜索：完整K扫描 + 固定K下仅优化alpha
METHODS = ["cfo", "gwo", "ala"]

# alpha采用变量特异搜索范围。
# SP已有完整搜索结果且平台alpha合理，因此保持0.5~10并继续复用。
# T2M在低K下多次触及0.5下边界，因此仅T2M把下限扩展至0.01。
# D2M暂时保持0.5~10。
# ALPHA_BOUNDS保留为函数默认/兼容值，正式搜索使用ALPHA_BOUNDS_BY_VARIABLE。
ALPHA_BOUNDS = (0.5, 10.0)
ALPHA_BOUNDS_BY_VARIABLE = {
    "SP": (0.5, 10.0),
    "T2M": (0.01, 10.0),
    "D2M": (0.5, 10.0),
}

# K固定为8个离散候选：2,3,...,9。
# 每个固定K下，CFO/GWO/ALA只优化alpha。
K_CANDIDATES = tuple(range(2, 10))
K_BOUNDS = (min(K_CANDIDATES), max(K_CANDIDATES))

# 正式推荐版：一维alpha优化使用8个体、20代。
POP_SIZE = 8
MAX_ITER = 20

TAU = 0.0
DC = 0
INIT = 1
TOL = 1e-7
VMD_MAX_ITER = 500

# 不再使用人工百分比阈值早停。
# 这些兼容变量仅供旧函数/断点字段读取，实际均关闭。
OPTIMIZATION_EARLY_STOPPING = False
PATIENCE = 0
ALPHA_REL_IMPROVEMENT_THRESHOLD = 0.0
MIN_DELTA = 0.0

K_EARLY_STOPPING = False
K_REL_IMPROVEMENT_THRESHOLD = 0.0
K_PLATEAU_PATIENCE = 0

# 自动平台选择：
# 1) 若objective最小值位于搜索边界，使用归一化曲线的elbow/knee寻找收益递减起点；
# 2) 若最小值位于内部，联合最小化归一化objective和局部斜率；
# 3) K选择在内部情形额外加入较小K的复杂度偏好。
PLATEAU_SELECTION_METHOD = "automatic_low_flat_elbow_v1"

RANDOM_STATES = {
    "cfo": 20260620,
    "gwo": 20260621,
    "ala": 20260622,
}

# 每个固定K、每种优化器的理论最大candidate数。
MAX_CANDIDATES_PER_METHOD = {
    "cfo": POP_SIZE + 2 * POP_SIZE * MAX_ITER,
    "gwo": POP_SIZE + POP_SIZE * MAX_ITER,
    "ala": POP_SIZE + POP_SIZE * MAX_ITER,
}
# 这是“全部K都跑满且不早停”的理论上限，仅用于时间投影。
MAX_TOTAL_SEARCH_CANDIDATES = (
    len(SOURCE_SPECS)
    * len(K_CANDIDATES)
    * sum(MAX_CANDIDATES_PER_METHOD[m] for m in METHODS)
)

PAD_ODD = True

# 4. 300个空间分层格点 —— 不改
SEARCH_GRID_NY = 15
SEARCH_GRID_NX = 20
SEARCH_GRIDPOINT_NUMBER = SEARCH_GRID_NY * SEARCH_GRID_NX
assert SEARCH_GRIDPOINT_NUMBER == 300

# 5. 加速配置
# 5975WX = 32物理核 / 64逻辑线程。
# CPU线程固定32；Numba fallback直接在300格点矩阵上并行，
# 不再为每个格点启动一个Python VMD任务。
N_JOBS = 32
NUMBA_THREADS = 32
FFT_WORKERS = 32

# CPU fallback搜索阶段一次最多计算300条格点序列。
SEARCH_MATRIX_BATCH = 300

# 5a. candidate × grid 双矩阵 / CUDA配置
# auto: 有CUDA时使用GPU，否则CPU fallback；正式一周级目标建议保持cuda。
COMPUTE_BACKEND = "cuda"
CUDA_DEVICE = 0

# CPU侧仅负责NetCDF、FFT预处理和fallback，限制32线程保证桌面可用。
TORCH_CPU_THREADS = 4

# 一个GPU batch同时包含多个(candidate, gridpoint) task。
# 根据K和当前可用显存自动缩放；发生CUDA OOM时自动二分重试。
GPU_TASK_BATCH_MIN = 16
GPU_TASK_BATCH_MAX = 192
GPU_WORKSPACE_FRACTION = 0.62
GPU_WORKSPACE_TARGET_GB = 22.0

# 主内存安全线：不强制占满512 GB，始终至少给系统/文件缓存保留256 GB。
HOST_RAM_RESERVE_GB = 256.0
HOST_RAM_SOFT_LIMIT_GB = 512.0

# GPU envelope entropy仍以float64聚合，但按时间块转换，避免巨型float64副本。
ENTROPY_TIME_CHUNK = 16384
GPU_ENTROPY_FLOAT64 = False  # float32 envelope-entropy accumulation

# full-domain和最终分解一次覆盖整个49×73空间域；
# 1 TB RAM足够，避免8×8小tile重复读取/FFT。
FULL_DOMAIN_WHOLE_GRID = True


# 兼容旧函数的tile参数；新Cell 7/10实际一次处理整个49×73空间域。
TILE_Y = 49
TILE_X = 73

# 搜索300点时，按31天时间块从NetCDF批量读取所有300点，
# 避免“300个格点逐点读取20多年”的低效随机I/O。
LOAD_TIME_CHUNK = 744

# 快速VMD内部采用float32/complex64；
# 目标函数最终仍以float64聚合。
# 这是数值实现精度优化，不改变VMD公式、alpha/K、tol或目标定义。
FAST_REAL_DTYPE = np.float32
FAST_COMPLEX_DTYPE = np.complex64
FAST_SOLVER_SIGNATURE = "candidate-grid-cuda-fixedK-alpha1d-autoplateau-v3"

# Numba fastmath仅改变底层浮点运算优化，不改变算法定义。
NUMBA_FASTMATH = True

# 6. NetCDF输出
COMPRESSION_LEVEL = 9
TIME_CHUNK = 168  # 7天
SYNC_EVERY_TILES = 4  # 每4个tile强制sync一次；崩溃时最多重算3个tile
OVERWRITE_OUTPUTS = False
RESUME_DECOMPOSITION = True
SKIP_COMPLETE_SPLIT = True

# 7. 长任务断点续跑
CHECKPOINT_DIR = DATA_DIR / "VMD_checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

SEARCH_RESULT_JSON = DATA_DIR / "VMD_search_K2-9_alpha_autoplateau_300grid_results.json"
FULL_REEVAL_JSON = DATA_DIR / "VMD_K2-9_autoplateau_full_domain_reevaluation.json"
FINAL_SELECTION_JSON = DATA_DIR / "VMD_K2-9_autoplateau_final_selection.json"

PARAM_XLSX = DATA_DIR / "VMD_K2-9_autoplateau_300search_fullgrid_parameter_summary.xlsx"
SPLIT_CSV = DATA_DIR / "VMD_split_information.csv"
TABLE_S4_CSV = DATA_DIR / "VMD_K2-9_autoplateau_TableS4_final_values.csv"

REUSE_FINISHED_SEARCH = True
REUSE_FINISHED_FULL_REEVALUATION = True
RESUME_OPTIMIZER_ITERATIONS = True

# True：optimizer正常完成后也保留search_{variable}_{optimizer}_K*.pkl。
# 这样即使完成optimizer后、JSON写入前程序异常退出，也可以从checkpoint恢复，
# 不需要重新计算该optimizer。
KEEP_COMPLETED_OPTIMIZER_CHECKPOINTS = True

# 8. 固定Numba线程数
_available_cpu = os.cpu_count() or 1
NUMBA_THREADS = min(int(NUMBA_THREADS), int(_available_cpu))
FFT_WORKERS = min(int(FFT_WORKERS), int(_available_cpu))
set_num_threads(NUMBA_THREADS)

# 当前快速kernel针对论文固定设置做专门化。
if not (float(TAU) == 0.0 and int(DC) == 0 and int(INIT) == 1):
    raise ValueError(
        "当前矩阵加速kernel针对论文固定设置TAU=0、DC=0、INIT=1。"
        "如果修改这些方法参数，需要同时修改快速kernel。"
    )

print("DATA_DIR:", DATA_DIR)
print("Search gridpoints:", SEARCH_GRIDPOINT_NUMBER)
print("K candidates:", K_CANDIDATES)
print("Alpha bounds by variable:", ALPHA_BOUNDS_BY_VARIABLE)
print("Population / max iterations:", POP_SIZE, "/", MAX_ITER)
print("Optimizer early stopping:", OPTIMIZATION_EARLY_STOPPING)
print("K early stopping:", K_EARLY_STOPPING)
print("Automatic plateau selector:", PLATEAU_SELECTION_METHOD)
print("Numba version:", numba.__version__)
print("Numba threads:", get_num_threads())
print("FFT workers:", FFT_WORKERS)
print("Search matrix batch:", SEARCH_MATRIX_BATCH)
print("Fast solver:", FAST_SOLVER_SIGNATURE)
print("Compression level:", COMPRESSION_LEVEL)


In [ ]:
# Cell 2. NetCDF读取、维度识别、原模型split精确复现

def detect_dimensions(nc, var_name):
    """
    识别目标变量的time/latitude/longitude维度。
    本任务要求变量为3D，但维度排列顺序可以不同。
    """
    if var_name not in nc.variables:
        raise KeyError(
            f"变量 {var_name!r} 不在文件中。已有变量：{list(nc.variables.keys())}"
        )

    var = nc.variables[var_name]
    dims = list(var.dimensions)

    time_aliases = {"time", "valid_time"}
    lat_aliases = {"latitude", "lat"}
    lon_aliases = {"longitude", "lon"}

    def pick(alias_set, label):
        found = [d for d in dims if d.lower() in alias_set]
        if len(found) != 1:
            raise ValueError(
                f"{var_name}: 无法唯一识别{label}维。变量维度={dims}"
            )
        return found[0]

    time_dim = pick(time_aliases, "time")
    lat_dim = pick(lat_aliases, "latitude")
    lon_dim = pick(lon_aliases, "longitude")

    if len(dims) != 3 or set(dims) != {time_dim, lat_dim, lon_dim}:
        raise ValueError(
            f"{var_name}维度为{dims}。本Notebook要求仅包含time/lat/lon三个维度。"
        )

    return {
        "var": var,
        "dims": dims,
        "time_dim": time_dim,
        "lat_dim": lat_dim,
        "lon_dim": lon_dim,
        "time_axis": dims.index(time_dim),
        "lat_axis": dims.index(lat_dim),
        "lon_axis": dims.index(lon_dim),
    }


def read_tyx(var, meta, time_slice, y_slice=slice(None), x_slice=slice(None)):
    """
    从任意维度顺序读取，并统一返回(time, lat, lon)。
    """
    idx = [slice(None)] * 3
    idx[meta["time_axis"]] = time_slice
    idx[meta["lat_axis"]] = y_slice
    idx[meta["lon_axis"]] = x_slice

    arr = var[tuple(idx)]
    arr = np.ma.filled(arr, np.nan)
    arr = np.asarray(arr, dtype=np.float64)

    arr = np.transpose(
        arr,
        axes=[
            meta["time_axis"],
            meta["lat_axis"],
            meta["lon_axis"],
        ],
    )
    return arr


def read_point_series(var, meta, time_slice, iy, ix):
    a = read_tyx(
        var,
        meta,
        time_slice=time_slice,
        y_slice=slice(iy, iy + 1),
        x_slice=slice(ix, ix + 1),
    )
    return a[:, 0, 0]


def get_model_aligned_split_slices(
    n_raw_time,
    timestep=6,
    test_size=0.2,
    valid_size=0.1,
):
    """
    严格复现上传CNN-LSTM notebook：

      vx = predictor[:-1]            -> 长度 n_raw_time - 1
      timestep = 6                   -> 样本数 n_raw_time - 6
      test: 后20%
      valid: 从前80%中再切总样本量10%
             原函数公式：
             int((1-valid_size/(1-test_size))*trainval_count)

    返回两个层次：
      sample_slices: 模型样本轴上的train/valid/test
      predictor_slices: 每个split独立构造6小时输入时，需要的原始predictor时间范围

    predictor_slices之间重叠timestep-1=5小时。
    这是为了让每个split独立VMD后仍能单独构造完整6小时样本。
    """
    n_predictor_time = n_raw_time - 1
    n_samples = n_predictor_time - timestep + 1

    if n_samples <= 0:
        raise ValueError("原始时间长度不足以构造timestep窗口。")

    test_start = int((1.0 - test_size) * n_samples)

    train_end = int(
        (1.0 - valid_size / (1.0 - test_size))
        * test_start
    )

    sample_slices = {
        "train": slice(0, train_end),
        "valid": slice(train_end, test_start),
        "test": slice(test_start, n_samples),
    }

    # 样本[a,b)需要predictor原始索引[a, b+timestep-1)
    predictor_slices = {
        name: slice(slc.start, slc.stop + timestep - 1)
        for name, slc in sample_slices.items()
    }

    # 边界检查
    assert predictor_slices["train"].stop <= n_predictor_time
    assert predictor_slices["valid"].stop <= n_predictor_time
    assert predictor_slices["test"].stop == n_predictor_time

    # 每个split单独建窗口后，样本数应完全恢复
    for name in ["train", "valid", "test"]:
        n_time = predictor_slices[name].stop - predictor_slices[name].start
        recovered_samples = n_time - timestep + 1
        expected_samples = sample_slices[name].stop - sample_slices[name].start
        assert recovered_samples == expected_samples

    info = {
        "n_raw_time": int(n_raw_time),
        "n_predictor_time": int(n_predictor_time),
        "n_samples": int(n_samples),

        "train_sample_start": int(sample_slices["train"].start),
        "train_sample_end_exclusive": int(sample_slices["train"].stop),
        "valid_sample_start": int(sample_slices["valid"].start),
        "valid_sample_end_exclusive": int(sample_slices["valid"].stop),
        "test_sample_start": int(sample_slices["test"].start),
        "test_sample_end_exclusive": int(sample_slices["test"].stop),

        "n_train_samples": int(
            sample_slices["train"].stop - sample_slices["train"].start
        ),
        "n_valid_samples": int(
            sample_slices["valid"].stop - sample_slices["valid"].start
        ),
        "n_test_samples": int(
            sample_slices["test"].stop - sample_slices["test"].start
        ),

        "history_context_hours": int(timestep - 1),
    }

    for name in ["train", "valid", "test"]:
        info[f"{name}_predictor_start"] = int(predictor_slices[name].start)
        info[f"{name}_predictor_end_exclusive"] = int(predictor_slices[name].stop)
        info[f"{name}_predictor_time_length"] = int(
            predictor_slices[name].stop - predictor_slices[name].start
        )

    return sample_slices, predictor_slices, info


def decode_time_at(nc, time_dim, index):
    tv = nc.variables[time_dim]
    value = tv[index]
    units = getattr(tv, "units", None)
    calendar = getattr(tv, "calendar", "standard")

    if units is None:
        return str(value)

    try:
        return str(num2date(value, units=units, calendar=calendar))
    except Exception:
        return str(value)


# 检查三个源文件网格和时间轴长度
reference_signature = None
sample_slices = None
predictor_slices = None
split_info = None

for key, spec in SOURCE_SPECS.items():
    path = SOURCE_DIR / spec["filename"]

    if not path.exists():
        raise FileNotFoundError(f"源文件不存在：{path}")

    with Dataset(path, "r") as nc:
        meta = detect_dimensions(nc, spec["var_name"])
        ntime = len(nc.dimensions[meta["time_dim"]])
        ny = len(nc.dimensions[meta["lat_dim"]])
        nx = len(nc.dimensions[meta["lon_dim"]])

        sig = (ntime, ny, nx)

        if reference_signature is None:
            reference_signature = sig
            sample_slices, predictor_slices, split_info = get_model_aligned_split_slices(
                n_raw_time=ntime,
                timestep=TIMESTEP,
                test_size=TEST_SIZE,
                valid_size=VALID_SIZE,
            )
        elif sig != reference_signature:
            raise ValueError(
                f"三个输入文件的(time,lat,lon)大小不一致："
                f"reference={reference_signature}, {path.name}={sig}"
            )

        print(
            f"{key}: {path.name} | var={spec['var_name']} | "
            f"shape(time,lat,lon)={sig}"
        )

split_df = pd.DataFrame([split_info])
split_df.to_csv(SPLIT_CSV, index=False, encoding="utf-8-sig")

# 打印样本和时间范围
sp_spec = SOURCE_SPECS["SP"]
with Dataset(SOURCE_DIR / sp_spec["filename"], "r") as nc:
    meta = detect_dimensions(nc, sp_spec["var_name"])
    td = meta["time_dim"]

    print("\n" + "=" * 105)
    print("原CNN-LSTM样本级split复现：")
    print(
        f"total samples={split_info['n_samples']}, "
        f"train={split_info['n_train_samples']}, "
        f"valid={split_info['n_valid_samples']}, "
        f"test={split_info['n_test_samples']}"
    )

    print("\n每个split独立VMD使用的predictor时间范围：")
    for name in ["train", "valid", "test"]:
        slc = predictor_slices[name]
        print(
            f"{name:5s}: raw predictor index [{slc.start}, {slc.stop}) | "
            f"Ntime={slc.stop-slc.start} | "
            f"{decode_time_at(nc, td, slc.start)} -> "
            f"{decode_time_at(nc, td, slc.stop-1)}"
        )

    print(
        f"\n相邻split会重叠 {TIMESTEP-1} 个历史小时，"
        "但三个split分别独立VMD。"
    )
    print("=" * 105)


In [ ]:
# Cell 3. 参考VMD + 矩阵/Numba快速VMD + 包络熵

# A. 原始单序列float64实现：保留用于数值QC，不再用于正式大规模搜索
def vmd_memory_efficient(
    signal,
    alpha,
    K,
    tau=0.0,
    DC=0,
    init=1,
    tol=1e-7,
    max_iter=500,
    pad_odd=True,
    random_state=0,
):
    """单序列float64参考实现；数学逻辑与原Notebook一致。"""
    f = np.asarray(signal, dtype=np.float64).squeeze()

    if f.ndim != 1:
        raise ValueError("VMD输入必须是一维时间序列。")
    if len(f) < 4:
        raise ValueError("VMD输入序列过短。")
    if not np.all(np.isfinite(f)):
        raise ValueError("VMD输入包含NaN/Inf。")

    original_len = len(f)

    if pad_odd and (len(f) % 2 == 1):
        f_work = np.concatenate([f, f[-1:]])
    else:
        f_work = f.copy()

    n0 = len(f_work)
    half = n0 // 2

    f_mirror = np.concatenate([
        f_work[:half][::-1],
        f_work,
        f_work[-half:][::-1],
    ])

    T = len(f_mirror)
    freqs = np.arange(1, T + 1, dtype=np.float64) / T - 0.5 - 1.0 / T

    f_hat = np.fft.fftshift(np.fft.fft(f_mirror))
    f_hat_plus = f_hat.copy()
    f_hat_plus[: T // 2] = 0

    K = int(K)
    alpha_vec = float(alpha) * np.ones(K, dtype=np.float64)
    rng = np.random.default_rng(random_state)

    omega_prev = np.zeros(K, dtype=np.float64)

    if int(init) == 1:
        omega_prev = (0.5 / K) * np.arange(K, dtype=np.float64)
    elif int(init) == 2:
        fs = 1.0 / n0
        omega_prev = np.sort(
            np.exp(
                np.log(fs)
                + (np.log(0.5) - np.log(fs)) * rng.random(K)
            )
        )
    elif int(init) == 0:
        omega_prev[:] = 0.0
    else:
        raise ValueError("init只能为0、1或2。")

    if int(DC):
        omega_prev[0] = 0.0

    omega_curr = omega_prev.copy()
    u_prev = np.zeros((T, K), dtype=np.complex128)
    u_curr = np.zeros((T, K), dtype=np.complex128)
    lambda_hat = np.zeros(T, dtype=np.complex128)

    eps = np.finfo(np.float64).eps
    u_diff = float(tol) + eps

    positive_slice = slice(T // 2, T)
    positive_freqs = freqs[positive_slice]

    n_iter_done = 0

    while (u_diff > tol) and (n_iter_done < int(max_iter)):
        sum_prev = np.sum(u_prev, axis=1)
        sum_uk = sum_prev - u_prev[:, 0]

        for k in range(K):
            if k > 0:
                sum_uk = sum_uk + u_curr[:, k - 1] - u_prev[:, k]

            denom = 1.0 + alpha_vec[k] * (freqs - omega_prev[k]) ** 2

            u_curr[:, k] = (
                f_hat_plus - sum_uk - lambda_hat / 2.0
            ) / denom

            if not (int(DC) and k == 0):
                power = np.abs(u_curr[positive_slice, k]) ** 2
                power_sum = np.sum(power)

                if np.isfinite(power_sum) and power_sum > eps:
                    omega_curr[k] = (
                        np.sum(positive_freqs * power) / power_sum
                    )
                else:
                    omega_curr[k] = omega_prev[k]
            else:
                omega_curr[k] = 0.0

        lambda_hat = (
            lambda_hat
            + float(tau) * (np.sum(u_curr, axis=1) - f_hat_plus)
        )

        u_diff = eps
        for k in range(K):
            d = u_curr[:, k] - u_prev[:, k]
            u_diff += np.vdot(d, d).real / T

        u_diff = float(abs(u_diff))
        n_iter_done += 1
        u_prev, u_curr = u_curr, u_prev
        omega_prev, omega_curr = omega_curr, omega_prev

    final_u_hat_plus = u_prev

    u_hat = np.zeros((T, K), dtype=np.complex128)
    u_hat[T // 2 : T, :] = final_u_hat_plus[T // 2 : T, :]

    idxs = np.flip(np.arange(1, T // 2 + 1))
    u_hat[idxs, :] = np.conj(final_u_hat_plus[T // 2 : T, :])
    u_hat[0, :] = np.conj(u_hat[-1, :])

    modes_mirror = np.empty((K, T), dtype=np.float64)

    for k in range(K):
        modes_mirror[k, :] = np.real(
            np.fft.ifft(np.fft.ifftshift(u_hat[:, k]))
        )

    modes = modes_mirror[:, T // 4 : 3 * T // 4]
    modes = modes[:, :original_len]
    return modes


def envelope_entropy_1d(x, eps=1e-12):
    x = np.asarray(x, dtype=np.float64)

    if x.ndim != 1 or len(x) < 3:
        return np.inf
    if not np.all(np.isfinite(x)):
        return np.inf

    envelope = np.abs(hilbert(x))
    s = np.sum(envelope)

    if (not np.isfinite(s)) or s <= eps:
        return np.inf

    p = envelope / (s + eps)
    p = np.clip(p, eps, 1.0)

    ent = -np.sum(p * np.log(p))
    ent = ent / np.log(len(p) + eps)
    return float(ent)


def one_series_vmd_objective(
    series,
    alpha,
    K,
    tau=TAU,
    DC=DC,
    init=INIT,
    tol=TOL,
    max_iter=VMD_MAX_ITER,
    pad_odd=PAD_ODD,
    eps=1e-12,
):
    """原单序列objective，仅用于小规模QC/回归检查。"""
    try:
        x = np.asarray(series, dtype=np.float64)

        if x.ndim != 1 or len(x) < 10:
            return np.inf
        if not np.all(np.isfinite(x)):
            return np.inf
        if np.nanstd(x) <= eps:
            return np.inf

        imfs = vmd_memory_efficient(
            x,
            alpha=alpha,
            K=K,
            tau=tau,
            DC=DC,
            init=init,
            tol=tol,
            max_iter=max_iter,
            pad_odd=pad_odd,
        )

        scores = []
        for k in range(int(K)):
            imf = imfs[k]

            if not np.all(np.isfinite(imf)):
                continue
            if np.sum(np.abs(np.diff(imf))) <= eps:
                continue

            e = envelope_entropy_1d(imf, eps=eps)
            if np.isfinite(e):
                scores.append(e)

        if len(scores) == 0:
            return np.inf

        return float(np.mean(scores))
    except Exception:
        return np.inf


# B. 快速矩阵VMD kernel
@njit(parallel=True, fastmath=NUMBA_FASTMATH, cache=False)
def _vmd_positive_batch_numba(
    f_hat_pos,
    freqs_pos,
    alpha,
    K,
    tol,
    max_iter,
    T_full,
):
    """
    对B条时间序列同时执行VMD。

    加速点：
    1) 只保存正频率半谱；原实现负频率半谱始终为0。
    2) 每条序列只保留一个u矩阵，Gauss-Seidel原位更新。
    3) sum(u)在迭代间递推，不重复对K求和。
    4) 收敛差在更新u时同步累计，不再额外遍历一次频谱。
    5) B条格点序列通过Numba prange在CPU线程上并行。

    针对本论文固定TAU=0、DC=0、INIT=1。
    数学更新次序与原单序列实现一致。
    """
    B, P = f_hat_pos.shape
    out = np.zeros((B, P, K), dtype=np.complex64)
    n_iters = np.zeros(B, dtype=np.int32)

    alpha32 = np.float32(alpha)
    tol32 = np.float32(tol)
    inv_T = np.float32(1.0 / T_full)
    tiny = np.float32(1e-30)

    for b in prange(B):
        u = np.zeros((P, K), dtype=np.complex64)
        omega = np.empty(K, dtype=np.float32)

        for k in range(K):
            omega[k] = np.float32((0.5 / K) * k)

        sum_prev = np.zeros(P, dtype=np.complex64)
        sum_curr = np.zeros(P, dtype=np.complex64)
        sum_uk = np.zeros(P, dtype=np.complex64)

        # 不把float32 machine epsilon加到u_diff中；
        # 否则eps≈1.19e-7会大于论文tol=1e-7而强制跑满500轮。
        # 这里直接按数学定义计算变化量，与tol比较。
        u_diff = tol32 + np.float32(1.0)
        n_iter_done = 0

        while (u_diff > tol32) and (n_iter_done < max_iter):
            for j in range(P):
                sum_uk[j] = sum_prev[j] - u[j, 0]
                sum_curr[j] = np.complex64(0.0 + 0.0j)

            u_diff = np.float32(0.0)

            for k in range(K):
                if k > 0:
                    for j in range(P):
                        sum_uk[j] = (
                            sum_uk[j]
                            + u[j, k - 1]
                            - u[j, k]
                        )

                om = omega[k]
                power_sum = np.float32(0.0)
                weighted_power_sum = np.float32(0.0)

                for j in range(P):
                    old = u[j, k]
                    df = freqs_pos[j] - om
                    denom = np.float32(1.0) + alpha32 * df * df

                    z = (f_hat_pos[b, j] - sum_uk[j]) / denom
                    u[j, k] = z
                    sum_curr[j] += z

                    d = z - old
                    u_diff += (
                        d.real * d.real + d.imag * d.imag
                    ) * inv_T

                    pw = z.real * z.real + z.imag * z.imag
                    power_sum += pw
                    weighted_power_sum += freqs_pos[j] * pw

                if power_sum > tiny and np.isfinite(power_sum):
                    omega[k] = weighted_power_sum / power_sum

            n_iter_done += 1
            tmp = sum_prev
            sum_prev = sum_curr
            sum_curr = tmp

        for j in range(P):
            for k in range(K):
                out[b, j, k] = u[j, k]

        n_iters[b] = n_iter_done

    return out, n_iters


def prepare_vmd_spectra_matrix(series_matrix, fft_workers=FFT_WORKERS):
    """
    对一组格点时间序列只做一次：odd padding + mirror + FFT。
    搜索阶段同一predictor的所有candidate重复使用该频谱，
    因而不再为每个(alpha,K)重复做300次镜像和FFT。
    """
    x = np.ascontiguousarray(series_matrix, dtype=FAST_REAL_DTYPE)

    if x.ndim != 2:
        raise ValueError("series_matrix必须为(grid, time)二维数组。")
    if x.shape[1] < 4:
        raise ValueError("时间长度过短。")
    if not np.all(np.isfinite(x)):
        raise ValueError("矩阵VMD输入包含NaN/Inf；请先过滤无效格点。")

    original_len = int(x.shape[1])

    if PAD_ODD and (x.shape[1] % 2 == 1):
        x_work = np.concatenate([x, x[:, -1:]], axis=1)
    else:
        x_work = x

    n0 = int(x_work.shape[1])
    half = n0 // 2

    mirrored = np.concatenate(
        [
            x_work[:, :half][:, ::-1],
            x_work,
            x_work[:, -half:][:, ::-1],
        ],
        axis=1,
    )
    mirrored = np.ascontiguousarray(mirrored, dtype=FAST_REAL_DTYPE)

    T_full = int(mirrored.shape[1])

    # scipy.fft对float32保留complex64，且支持workers并行。
    f_hat = sp_fft.fft(
        mirrored,
        axis=1,
        workers=int(fft_workers),
        overwrite_x=True,
    )
    f_hat = sp_fft.fftshift(f_hat, axes=1)

    f_hat_pos = np.ascontiguousarray(
        f_hat[:, T_full // 2 :],
        dtype=FAST_COMPLEX_DTYPE,
    )

    freqs_pos = (
        np.arange(0, T_full // 2, dtype=np.float32)
        / np.float32(T_full)
    )
    # 与原式 arange(1,T+1)/T - 0.5 - 1/T 在正半谱完全等价：
    # 正半谱频率 = 0/T, 1/T, ..., (T/2-1)/T
    freqs_pos = np.ascontiguousarray(freqs_pos, dtype=np.float32)

    del mirrored, f_hat
    if x_work is not x:
        del x_work
    gc.collect()

    return {
        "f_hat_pos": f_hat_pos,
        "freqs_pos": freqs_pos,
        "T_full": T_full,
        "original_len": original_len,
        "n_series": int(x.shape[0]),
    }


def _analytic_signal_matrix(x, workers=FFT_WORKERS):
    """与scipy.signal.hilbert(axis=-1)等价，但显式使用FFT workers。"""
    x = np.ascontiguousarray(x, dtype=np.float32)
    N = int(x.shape[1])

    xf = sp_fft.fft(x, axis=1, workers=int(workers))
    h = np.zeros(N, dtype=np.float32)

    if N % 2 == 0:
        h[0] = 1.0
        h[N // 2] = 1.0
        h[1 : N // 2] = 2.0
    else:
        h[0] = 1.0
        h[1 : (N + 1) // 2] = 2.0

    analytic = sp_fft.ifft(
        xf * h[None, :],
        axis=1,
        workers=int(workers),
        overwrite_x=True,
    )
    return analytic


def reconstruct_mode_matrix_from_positive(
    positive_mode_spectrum,
    original_len,
    T_full,
    fft_workers=FFT_WORKERS,
):
    """
    从VMD正半谱重构B条实值IMF。

    使用irfft替代原来的“构造完整共轭谱 + ifft + real”，
    数学结果等价，但内存和FFT计算量约减半。
    """
    pos = np.ascontiguousarray(
        positive_mode_spectrum,
        dtype=FAST_COMPLEX_DTYPE,
    )
    B, P = pos.shape

    coeff = np.empty((B, P + 1), dtype=FAST_COMPLEX_DTYPE)
    coeff[:, :P] = pos
    coeff[:, P] = np.conj(pos[:, -1])

    mirrored_modes = sp_fft.irfft(
        coeff,
        n=int(T_full),
        axis=1,
        workers=int(fft_workers),
        overwrite_x=True,
    )

    modes = mirrored_modes[
        :,
        T_full // 4 : 3 * T_full // 4,
    ]
    modes = modes[:, : int(original_len)]
    return np.ascontiguousarray(modes, dtype=np.float32)


def objective_scores_from_final_positive(
    final_positive,
    original_len,
    T_full,
    eps=1e-12,
):
    """
    final_positive: (B, positive_frequency, K)

    完全保持论文objective的聚合层次：
      每格点：有效IMF envelope entropy平均
      全局：再对有效格点平均（在外层完成）
    """
    B, _, K = final_positive.shape
    score_sums = np.zeros(B, dtype=np.float64)
    score_counts = np.zeros(B, dtype=np.int32)

    for k in range(int(K)):
        modes = reconstruct_mode_matrix_from_positive(
            final_positive[:, :, k],
            original_len=original_len,
            T_full=T_full,
            fft_workers=FFT_WORKERS,
        )

        variation = np.sum(
            np.abs(np.diff(modes, axis=1)),
            axis=1,
            dtype=np.float64,
        )

        valid = (
            np.all(np.isfinite(modes), axis=1)
            & np.isfinite(variation)
            & (variation > eps)
        )

        if np.any(valid):
            idx = np.flatnonzero(valid)
            analytic = _analytic_signal_matrix(
                modes[idx],
                workers=FFT_WORKERS,
            )
            envelope = np.abs(analytic)

            s = np.sum(envelope, axis=1, dtype=np.float64)
            good_s = np.isfinite(s) & (s > eps)

            if np.any(good_s):
                good_idx = idx[good_s]
                env_good = envelope[good_s]
                s_good = s[good_s]

                # 保持和原函数相同的p/clip/log定义。
                p = env_good / (s_good[:, None] + eps)
                p = np.clip(p, eps, 1.0)

                ent = -np.sum(
                    p * np.log(p),
                    axis=1,
                    dtype=np.float64,
                )
                ent = ent / np.log(float(original_len) + eps)

                finite_ent = np.isfinite(ent)
                if np.any(finite_ent):
                    gi = good_idx[finite_ent]
                    score_sums[gi] += ent[finite_ent]
                    score_counts[gi] += 1

            del analytic, envelope

        del modes

    scores = np.full(B, np.inf, dtype=np.float64)
    good = score_counts > 0
    scores[good] = score_sums[good] / score_counts[good]
    return scores


def fast_vmd_spectra_and_scores(prepared, alpha, K):
    """对一个prepared矩阵batch执行一次(alpha,K)VMD并返回格点objective。"""
    final_positive, n_iters = _vmd_positive_batch_numba(
        prepared["f_hat_pos"],
        prepared["freqs_pos"],
        float(alpha),
        int(K),
        float(TOL),
        int(VMD_MAX_ITER),
        int(prepared["T_full"]),
    )

    scores = objective_scores_from_final_positive(
        final_positive,
        original_len=prepared["original_len"],
        T_full=prepared["T_full"],
    )
    return final_positive, scores, n_iters


def slice_prepared(prepared, start, stop):
    """仅切格点轴；频率轴/时间信息共享。"""
    return {
        "f_hat_pos": np.ascontiguousarray(
            prepared["f_hat_pos"][start:stop]
        ),
        "freqs_pos": prepared["freqs_pos"],
        "T_full": prepared["T_full"],
        "original_len": prepared["original_len"],
        "n_series": int(stop - start),
    }


# D. CUDA candidate×grid double-matrix engine
try:
    import torch
except Exception as e:
    torch = None
    _TORCH_IMPORT_ERROR = e
else:
    _TORCH_IMPORT_ERROR = None

try:
    import psutil
except Exception:
    psutil = None


def host_memory_guard(required_bytes, label="allocation"):
    """在大RAM分配前检查可用内存；外部程序占用过高时宁可停止也不硬顶OOM。"""
    required_gb = float(required_bytes) / (1024.0 ** 3)
    if required_gb > float(HOST_RAM_SOFT_LIMIT_GB):
        raise MemoryError(
            f"{label}: 单次预计内存{required_gb:.1f} GiB超过软件安全上限"
            f"{HOST_RAM_SOFT_LIMIT_GB:.1f} GiB。"
        )

    if psutil is not None:
        available_gb = float(psutil.virtual_memory().available) / (1024.0 ** 3)
        if required_gb + float(HOST_RAM_RESERVE_GB) > available_gb:
            raise MemoryError(
                f"{label}: 预计需要{required_gb:.1f} GiB，但当前仅有"
                f"{available_gb:.1f} GiB可用；代码要求额外保留"
                f"{HOST_RAM_RESERVE_GB:.1f} GiB给Windows/Jupyter。"
            )
    return required_gb


def _resolve_compute_device():
    if str(COMPUTE_BACKEND).lower() == "cpu":
        return "cpu"

    if torch is None:
        if str(COMPUTE_BACKEND).lower() == "cuda":
            raise ImportError(
                "COMPUTE_BACKEND='cuda'，但当前环境无法import torch。"
            ) from _TORCH_IMPORT_ERROR
        return "cpu"

    if torch.cuda.is_available():
        return f"cuda:{int(CUDA_DEVICE)}"

    if str(COMPUTE_BACKEND).lower() == "cuda":
        raise RuntimeError(
            "COMPUTE_BACKEND='cuda'，但torch.cuda.is_available()为False。"
        )

    return "cpu"


COMPUTE_DEVICE = _resolve_compute_device()

if torch is not None:
    try:
        torch.set_num_threads(int(TORCH_CPU_THREADS))
    except Exception:
        pass

    if COMPUTE_DEVICE.startswith("cuda"):
        torch.cuda.set_device(int(CUDA_DEVICE))
        torch.backends.cudnn.benchmark = False


def prepared_to_torch_cache(prepared, device=COMPUTE_DEVICE):
    """
    将同一predictor已经预计算好的正半谱一次性放到GPU。

    搜索阶段：300个格点的f_hat_pos只上传一次，随后整个
    CFO/GWO/ALA搜索重复使用，不再重复CPU->GPU传输。
    """
    if torch is None:
        raise RuntimeError("torch不可用。")

    dev = torch.device(device)

    f_hat_pos = torch.as_tensor(
        np.ascontiguousarray(prepared["f_hat_pos"]),
        dtype=torch.complex64,
        device=dev,
    ).contiguous()

    freqs_pos = torch.as_tensor(
        np.ascontiguousarray(prepared["freqs_pos"]),
        dtype=torch.float32,
        device=dev,
    ).contiguous()

    return {
        "f_hat_pos": f_hat_pos,
        "freqs_pos": freqs_pos,
        "T_full": int(prepared["T_full"]),
        "original_len": int(prepared["original_len"]),
        "n_series": int(prepared["n_series"]),
        "device": str(dev),
    }


def free_torch_cache(cache):
    if cache is None:
        return
    try:
        del cache["f_hat_pos"]
        del cache["freqs_pos"]
    except Exception:
        pass
    if torch is not None and COMPUTE_DEVICE.startswith("cuda"):
        torch.cuda.empty_cache()


def _gpu_task_batch_size(P, K, requested_max=None):
    """
    根据当前GPU可用显存动态决定 candidate×grid task batch。

    估算故意偏保守，并且实际计算还带OOM自动二分重试，
    因此不会因为固定batch过大直接让Notebook崩掉。
    """
    if requested_max is None:
        requested_max = int(GPU_TASK_BATCH_MAX)

    if torch is None or not COMPUTE_DEVICE.startswith("cuda"):
        return max(1, min(int(requested_max), 32))

    free_bytes, total_bytes = torch.cuda.mem_get_info(int(CUDA_DEVICE))
    target_bytes = min(
        float(free_bytes) * float(GPU_WORKSPACE_FRACTION),
        float(GPU_WORKSPACE_TARGET_GB) * (1024.0 ** 3),
    )

    # 每task的主要内存：u(P,K) complex64 + fhat/old/z/sums/df/pw等工作区。
    # 额外系数取128 bytes/frequency，故意留有余量。
    bytes_per_task = float(P) * (8.0 * float(K) + 128.0)
    estimate = int(max(1.0, target_bytes / max(bytes_per_task, 1.0)))

    estimate = max(int(GPU_TASK_BATCH_MIN), estimate)
    estimate = min(int(requested_max), estimate)
    return int(max(1, estimate))


def _torch_vmd_task_batch(
    grid_f_hat_pos,
    freqs_pos,
    grid_indices,
    alpha_values,
    K,
    T_full,
    tol=TOL,
    max_iter=VMD_MAX_ITER,
):
    """
    同一个K下，对一批(candidate, gridpoint) task同时做VMD。

    每个task有自己的alpha和gridpoint；
    Gauss-Seidel的mode更新顺序、omega更新和tol判据不变。
    """
    device = grid_f_hat_pos.device

    if not torch.is_tensor(grid_indices):
        grid_indices = torch.as_tensor(
            grid_indices,
            dtype=torch.long,
            device=device,
        )
    else:
        grid_indices = grid_indices.to(device=device, dtype=torch.long)

    if not torch.is_tensor(alpha_values):
        alpha_values = torch.as_tensor(
            alpha_values,
            dtype=torch.float32,
            device=device,
        )
    else:
        alpha_values = alpha_values.to(device=device, dtype=torch.float32)

    f_hat = grid_f_hat_pos.index_select(0, grid_indices).contiguous()
    B, P = f_hat.shape
    K = int(K)

    u = torch.zeros(
        (B, K, P),
        dtype=torch.complex64,
        device=device,
    )

    omega0 = (
        (0.5 / float(K))
        * torch.arange(K, dtype=torch.float32, device=device)
    )
    omega = omega0.unsqueeze(0).expand(B, K).clone()

    sum_prev = torch.zeros((B, P), dtype=torch.complex64, device=device)
    sum_curr = torch.zeros_like(sum_prev)
    sum_uk = torch.zeros_like(sum_prev)

    active = torch.ones(B, dtype=torch.bool, device=device)
    n_iters = torch.zeros(B, dtype=torch.int32, device=device)

    inv_T = float(1.0 / float(T_full))
    tol_value = float(tol)
    tiny = 1e-30

    freq_row = freqs_pos.unsqueeze(0)
    alpha_col = alpha_values.unsqueeze(1)

    for _it in range(int(max_iter)):
        if not bool(torch.any(active).item()):
            break

        n_iters += active.to(torch.int32)

        sum_uk.copy_(sum_prev)
        sum_uk.sub_(u[:, 0, :])
        sum_curr.zero_()

        u_diff = torch.zeros(B, dtype=torch.float32, device=device)
        active_col = active.unsqueeze(1)

        for k in range(K):
            if k > 0:
                sum_uk.add_(u[:, k - 1, :])
                sum_uk.sub_(u[:, k, :])

            old = u[:, k, :].clone()
            om = omega[:, k].unsqueeze(1)
            df = freq_row - om
            denom = 1.0 + alpha_col * df * df
            z = (f_hat - sum_uk) / denom

            # 始终用active mask冻结已收敛task，避免每个mode都GPU->CPU同步。
            z = torch.where(active_col, z, old)

            u[:, k, :].copy_(z)
            sum_curr.add_(z)

            d = z - old
            u_diff.add_(
                torch.sum(
                    d.real * d.real + d.imag * d.imag,
                    dim=1,
                )
                * inv_T
            )

            pw = z.real * z.real + z.imag * z.imag
            power_sum = torch.sum(pw, dim=1)
            weighted = torch.sum(pw * freq_row, dim=1)

            good_power = (
                active
                & torch.isfinite(power_sum)
                & (power_sum > tiny)
            )
            omega_new = weighted / torch.clamp(power_sum, min=tiny)
            omega[:, k] = torch.where(
                good_power,
                omega_new,
                omega[:, k],
            )

        sum_prev.copy_(sum_curr)
        active = active & (u_diff > tol_value)

    # 转回(B,P,K)，与CPU fast solver接口一致。
    final_positive = u.permute(0, 2, 1).contiguous()
    return final_positive, n_iters


def _torch_hilbert_matrix(x):
    """torch版Hilbert analytic signal，axis=-1。"""
    N = int(x.shape[-1])
    xf = torch.fft.fft(x, dim=-1)
    h = torch.zeros(N, dtype=torch.float32, device=x.device)

    if N % 2 == 0:
        h[0] = 1.0
        h[N // 2] = 1.0
        h[1:N // 2] = 2.0
    else:
        h[0] = 1.0
        h[1:(N + 1) // 2] = 2.0

    return torch.fft.ifft(xf * h, dim=-1)


def _torch_objective_scores_from_positive(
    final_positive,
    original_len,
    T_full,
    eps=1e-12,
):
    """
    GPU上计算与论文相同的normalized envelope entropy objective。

    聚合层次不变：
      IMF entropy -> gridpoint内平均 -> candidate内300格点平均。
    """
    B, P, K = final_positive.shape
    device = final_positive.device

    score_sums = torch.zeros(B, dtype=torch.float64, device=device)
    score_counts = torch.zeros(B, dtype=torch.int32, device=device)
    logN = math.log(float(original_len) + float(eps))

    for k in range(int(K)):
        pos = final_positive[:, :, k]

        coeff = torch.empty(
            (B, P + 1),
            dtype=torch.complex64,
            device=device,
        )
        coeff[:, :P] = pos
        coeff[:, P] = torch.conj(pos[:, -1])

        mirrored_modes = torch.fft.irfft(
            coeff,
            n=int(T_full),
            dim=1,
        )
        modes = mirrored_modes[
            :,
            T_full // 4: 3 * T_full // 4,
        ][:, :int(original_len)].contiguous()

        variation = torch.sum(
            torch.abs(modes[:, 1:] - modes[:, :-1]),
            dim=1,
        )
        valid = (
            torch.isfinite(modes).all(dim=1)
            & torch.isfinite(variation)
            & (variation > float(eps))
        )

        if bool(torch.any(valid).item()):
            idx = torch.nonzero(valid, as_tuple=False).squeeze(1)
            analytic = _torch_hilbert_matrix(modes.index_select(0, idx))
            envelope = torch.abs(analytic)

            if GPU_ENTROPY_FLOAT64:
                # 严格float64累积路径：更接近CPU fast实现，但消费级GPU FP64较慢。
                s = torch.sum(envelope.to(torch.float64), dim=1)
                good_s = torch.isfinite(s) & (s > float(eps))

                if bool(torch.any(good_s).item()):
                    good_local = torch.nonzero(good_s, as_tuple=False).squeeze(1)
                    good_idx = idx.index_select(0, good_local)
                    env_good = envelope.index_select(0, good_local)
                    s_good = s.index_select(0, good_local)
                    ent = torch.zeros(good_idx.shape[0], dtype=torch.float64, device=device)

                    for t0 in range(0, int(original_len), int(ENTROPY_TIME_CHUNK)):
                        t1 = min(t0 + int(ENTROPY_TIME_CHUNK), int(original_len))
                        p = env_good[:, t0:t1].to(torch.float64) / (s_good[:, None] + float(eps))
                        p = torch.clamp(p, min=float(eps), max=1.0)
                        ent -= torch.sum(p * torch.log(p), dim=1)

                    ent = ent / logN
                    finite_ent = torch.isfinite(ent)
                    if bool(torch.any(finite_ent).item()):
                        gi = good_idx[finite_ent]
                        score_sums[gi] += ent[finite_ent]
                        score_counts[gi] += 1
            else:
                # GPU float32 envelope-entropy accumulation.
                # Cell 3会与float64参考objective自动比较，超过5e-5直接停止。
                s = torch.sum(envelope, dim=1)
                good_s = torch.isfinite(s) & (s > float(eps))

                if bool(torch.any(good_s).item()):
                    good_local = torch.nonzero(good_s, as_tuple=False).squeeze(1)
                    good_idx = idx.index_select(0, good_local)
                    env_good = envelope.index_select(0, good_local)
                    s_good = s.index_select(0, good_local)
                    ent32 = torch.zeros(good_idx.shape[0], dtype=torch.float32, device=device)

                    for t0 in range(0, int(original_len), int(ENTROPY_TIME_CHUNK)):
                        t1 = min(t0 + int(ENTROPY_TIME_CHUNK), int(original_len))
                        p = env_good[:, t0:t1] / (s_good[:, None] + float(eps))
                        p = torch.clamp(p, min=float(eps), max=1.0)
                        ent32 -= torch.sum(p * torch.log(p), dim=1)

                    ent32 = ent32 / float(logN)
                    finite_ent = torch.isfinite(ent32)
                    if bool(torch.any(finite_ent).item()):
                        gi = good_idx[finite_ent]
                        score_sums[gi] += ent32[finite_ent].to(torch.float64)
                        score_counts[gi] += 1

        del coeff, mirrored_modes, modes

    scores = torch.full(
        (B,),
        float("inf"),
        dtype=torch.float64,
        device=device,
    )
    good = score_counts > 0
    scores[good] = score_sums[good] / score_counts[good].to(torch.float64)
    return scores


def _run_gpu_task_chunk_with_retry(
    cache,
    grid_indices,
    alpha_values,
    K,
    requested_batch_size=None,
):
    """
    一个task chunk的VMD+objective。
    若CUDA OOM，自动把chunk二分重试，不让整个长任务直接崩掉。
    """
    if requested_batch_size is None:
        requested_batch_size = len(grid_indices)

    try:
        final_positive, n_iters = _torch_vmd_task_batch(
            cache["f_hat_pos"],
            cache["freqs_pos"],
            grid_indices,
            alpha_values,
            K=int(K),
            T_full=int(cache["T_full"]),
            tol=TOL,
            max_iter=VMD_MAX_ITER,
        )
        scores = _torch_objective_scores_from_positive(
            final_positive,
            original_len=int(cache["original_len"]),
            T_full=int(cache["T_full"]),
        )

        scores_np = scores.detach().cpu().numpy()
        n_iters_np = n_iters.detach().cpu().numpy()

        del final_positive, n_iters, scores
        return scores_np, n_iters_np

    except RuntimeError as e:
        msg = str(e).lower()
        if (
            COMPUTE_DEVICE.startswith("cuda")
            and "out of memory" in msg
            and len(grid_indices) > 1
        ):
            torch.cuda.empty_cache()
            mid = len(grid_indices) // 2
            s1, i1 = _run_gpu_task_chunk_with_retry(
                cache,
                grid_indices[:mid],
                alpha_values[:mid],
                K,
                requested_batch_size=max(1, mid),
            )
            s2, i2 = _run_gpu_task_chunk_with_retry(
                cache,
                grid_indices[mid:],
                alpha_values[mid:],
                K,
                requested_batch_size=max(1, len(grid_indices) - mid),
            )
            return (
                np.concatenate([s1, s2]),
                np.concatenate([i1, i2]),
            )
        raise


def candidate_objectives_population_matrix(
    cache,
    candidate_params,
    pbar=None,
    desc="candidate×grid matrix",
):
    """
    一次计算多个candidate在全部gridpoint上的objective。

    candidate_params: [(alpha, K), ...]

    关键加速：
      原来 candidate -> 300 grids 顺序循环；
      现在同一批candidate×grid构成任务矩阵，并按K分组，
      在GPU上以大batch同时计算。
    """
    candidate_params = [
        (float(a), int(k)) for a, k in candidate_params
    ]
    C = len(candidate_params)
    G = int(cache["n_series"])

    if C == 0:
        return np.empty(0, dtype=np.float64), {
            "mean_vmd_iterations": np.nan,
            "max_vmd_iterations": 0,
            "task_count": 0,
        }

    score_sums = np.zeros(C, dtype=np.float64)
    score_counts = np.zeros(C, dtype=np.int64)
    iter_sum = 0.0
    iter_count = 0
    iter_max = 0

    groups = {}
    for ci, (alpha, K) in enumerate(candidate_params):
        groups.setdefault(int(K), []).append((ci, float(alpha)))

    total_tasks = C * G
    completed_tasks = 0

    if pbar is not None:
        pbar.reset(total=total_tasks)
        pbar.set_description(desc, refresh=True)

    for K, entries in groups.items():
        candidate_ids = np.asarray([e[0] for e in entries], dtype=np.int64)
        alphas_group = np.asarray([e[1] for e in entries], dtype=np.float32)
        Cg = len(entries)
        n_tasks = Cg * G

        task_batch = _gpu_task_batch_size(
            P=int(cache["f_hat_pos"].shape[1]),
            K=int(K),
            requested_max=GPU_TASK_BATCH_MAX,
        )

        for start in range(0, n_tasks, task_batch):
            stop = min(start + task_batch, n_tasks)
            linear = np.arange(start, stop, dtype=np.int64)
            local_candidate = linear // G
            grid_idx = linear % G
            alpha_batch = alphas_group[local_candidate]

            scores, nit = _run_gpu_task_chunk_with_retry(
                cache,
                grid_idx,
                alpha_batch,
                K=int(K),
                requested_batch_size=task_batch,
            )

            finite = np.isfinite(scores)
            if np.any(finite):
                local_finite = local_candidate[finite]
                global_candidate = candidate_ids[local_finite]
                np.add.at(score_sums, global_candidate, scores[finite])
                np.add.at(score_counts, global_candidate, 1)

            if nit.size > 0:
                iter_sum += float(np.sum(nit, dtype=np.float64))
                iter_count += int(nit.size)
                iter_max = max(iter_max, int(np.max(nit)))

            completed = stop - start
            completed_tasks += completed
            if pbar is not None:
                pbar.update(completed)
                pbar.set_postfix_str(
                    f"K={K}, GPUtask={task_batch}, "
                    f"{completed_tasks}/{total_tasks} candidate-grid tasks",
                    refresh=True,
                )

    objectives = np.full(C, np.inf, dtype=np.float64)
    good = score_counts > 0
    objectives[good] = score_sums[good] / score_counts[good]

    stats = {
        "mean_vmd_iterations": (
            float(iter_sum / iter_count) if iter_count > 0 else np.nan
        ),
        "max_vmd_iterations": int(iter_max),
        "task_count": int(total_tasks),
        "candidate_count": int(C),
        "gridpoint_count": int(G),
    }
    return objectives, stats


def gpu_decompose_single_candidate_to_modes(
    cache,
    alpha,
    K,
    pbar=None,
    desc="GPU final VMD",
):
    """
    对cache中的全部gridpoint做单一(alpha,K)VMD，直接返回host端IMF：
      modes_host shape = (grid, K, time), float32

    大RAM机器上一次保存整个空间块的IMF，减少重复FFT/读盘。
    """
    G = int(cache["n_series"])
    N = int(cache["original_len"])
    K = int(K)
    P = int(cache["f_hat_pos"].shape[1])

    _host_imf_bytes = int(G) * int(K) * int(N) * np.dtype(np.float32).itemsize
    host_memory_guard(_host_imf_bytes, label="whole-domain IMF host cache")
    modes_host = np.empty((G, K, N), dtype=np.float32)
    task_batch = _gpu_task_batch_size(P=P, K=K, requested_max=GPU_TASK_BATCH_MAX)

    if pbar is not None:
        pbar.reset(total=G)
        pbar.set_description(desc, refresh=True)

    for start in range(0, G, task_batch):
        stop = min(start + task_batch, G)
        grid_idx = np.arange(start, stop, dtype=np.int64)
        alpha_batch = np.full(stop - start, float(alpha), dtype=np.float32)

        # OOM时对该batch自动二分；为了拿到IMF，需要手动递归函数。
        def _decompose_chunk(indices, alphas):
            try:
                fp, nit = _torch_vmd_task_batch(
                    cache["f_hat_pos"],
                    cache["freqs_pos"],
                    indices,
                    alphas,
                    K=K,
                    T_full=int(cache["T_full"]),
                    tol=TOL,
                    max_iter=VMD_MAX_ITER,
                )

                B, P2, K2 = fp.shape

                # ensemble安全对齐：
                # 按每个gridpoint最终模态的功率加权中心频率从低到高排序。
                # 这样不同alpha/optimizer在相同K下的IMF索引具有明确频率对应关系。
                pw = fp.real * fp.real + fp.imag * fp.imag
                freq_col = cache["freqs_pos"].view(1, P2, 1)
                power_sum = torch.sum(pw, dim=1)
                centers = (
                    torch.sum(pw * freq_col, dim=1)
                    / torch.clamp(power_sum, min=1e-30)
                )
                mode_order = torch.argsort(centers, dim=1)
                gather_idx = mode_order.unsqueeze(1).expand(B, P2, K2)
                fp = torch.gather(fp, dim=2, index=gather_idx)

                del pw, centers, power_sum, mode_order, gather_idx

                out = np.empty((B, K2, N), dtype=np.float32)

                for kk in range(K2):
                    pos = fp[:, :, kk]
                    coeff = torch.empty(
                        (B, P2 + 1),
                        dtype=torch.complex64,
                        device=fp.device,
                    )
                    coeff[:, :P2] = pos
                    coeff[:, P2] = torch.conj(pos[:, -1])
                    mm = torch.fft.irfft(
                        coeff,
                        n=int(cache["T_full"]),
                        dim=1,
                    )
                    mm = mm[
                        :,
                        cache["T_full"] // 4: 3 * cache["T_full"] // 4,
                    ][:, :N]
                    out[:, kk, :] = mm.detach().cpu().numpy().astype(np.float32, copy=False)
                    del coeff, mm

                del fp, nit
                return out

            except RuntimeError as e:
                if (
                    COMPUTE_DEVICE.startswith("cuda")
                    and "out of memory" in str(e).lower()
                    and len(indices) > 1
                ):
                    torch.cuda.empty_cache()
                    mid = len(indices) // 2
                    a = _decompose_chunk(indices[:mid], alphas[:mid])
                    b = _decompose_chunk(indices[mid:], alphas[mid:])
                    return np.concatenate([a, b], axis=0)
                raise

        out = _decompose_chunk(grid_idx, alpha_batch)
        modes_host[start:stop] = out
        del out

        if pbar is not None:
            pbar.update(stop - start)
            pbar.set_postfix_str(
                f"GPUtask={task_batch}, hostIMF={modes_host.nbytes/1024**3:.1f} GiB",
                refresh=True,
            )

    return modes_host


print("Compute backend:", COMPUTE_DEVICE)
if torch is not None:
    print("PyTorch version:", torch.__version__)
if COMPUTE_DEVICE.startswith("cuda"):
    props = torch.cuda.get_device_properties(int(CUDA_DEVICE))
    print("CUDA device:", props.name)
    print("CUDA VRAM GiB:", round(props.total_memory / 1024**3, 2))


# C. 小型数值等价自检 + Numba预编译
rng_test = np.random.default_rng(123)
tt = np.arange(0, 2048)
X_test = np.stack(
    [
        np.sin(2*np.pi*tt/64)
        + 0.35*np.sin(2*np.pi*tt/9)
        + 0.02*rng_test.standard_normal(len(tt)),
        np.sin(2*np.pi*tt/47)
        + 0.20*np.sin(2*np.pi*tt/13)
        + 0.02*rng_test.standard_normal(len(tt)),
    ],
    axis=0,
).astype(np.float32)

prepared_test = prepare_vmd_spectra_matrix(X_test, fft_workers=min(4, FFT_WORKERS))
_t0 = time.perf_counter()
fp_test, fast_scores_test, iter_test = fast_vmd_spectra_and_scores(
    prepared_test,
    alpha=2.0,
    K=4,
)
_compile_seconds = time.perf_counter() - _t0

ref_scores_test = np.array(
    [
        one_series_vmd_objective(X_test[i], alpha=2.0, K=4)
        for i in range(X_test.shape[0])
    ],
    dtype=np.float64,
)

max_abs_diff = float(np.max(np.abs(fast_scores_test - ref_scores_test)))
print("Fast VMD compile/self-test seconds:", round(_compile_seconds, 3))
print("Numba threading layer:", numba.threading_layer())
print("Numba active threads:", get_num_threads())
print("Fast VMD iteration counts:", iter_test.tolist())
print("Reference objectives:", ref_scores_test)
print("Fast matrix objectives:", fast_scores_test)
print("Max |fast-reference|:", max_abs_diff)

if not np.all(np.isfinite(fast_scores_test)):
    raise RuntimeError("快速矩阵VMD自检产生非有限objective。")
if max_abs_diff > 5e-5:
    raise RuntimeError(
        f"快速矩阵VMD与float64参考实现差异过大：{max_abs_diff}"
    )

# CUDA candidate×grid solver数值QC。
if COMPUTE_DEVICE.startswith("cuda"):
    _gpu_test_cache = prepared_to_torch_cache(prepared_test)
    _gpu_scores, _gpu_stats = candidate_objectives_population_matrix(
        _gpu_test_cache,
        [(2.0, 4)],
        pbar=None,
        desc="GPU QC",
    )
    _gpu_diff = float(abs(_gpu_scores[0] - float(np.mean(ref_scores_test))))
    print("GPU objective QC:", _gpu_scores.tolist())
    print("GPU |objective-reference|:", _gpu_diff)
    if _gpu_diff > 5e-5:
        raise RuntimeError(
            f"GPU candidate×grid objective与参考实现差异过大：{_gpu_diff}"
        )
    free_torch_cache(_gpu_test_cache)

# 重构QC
recon_test = np.zeros(X_test.shape[1], dtype=np.float64)
for k in range(4):
    m = reconstruct_mode_matrix_from_positive(
        fp_test[0, :, k][None, :],
        original_len=X_test.shape[1],
        T_full=prepared_test["T_full"],
        fft_workers=min(4, FFT_WORKERS),
    )[0]
    recon_test += m.astype(np.float64)

print(
    "Fast VMD reconstruction RMSE:",
    float(np.sqrt(np.mean((recon_test - X_test[0]) ** 2))),
)

del prepared_test, fp_test, X_test, recon_test
gc.collect()


In [ ]:
# Cell 4. 300个空间分层格点读取 + 矩阵candidate objective


def stratified_300_indices(ny, nx):
    """15×20空间分层规则点，共300点。"""
    ys = np.unique(
        np.rint(np.linspace(0, ny - 1, SEARCH_GRID_NY)).astype(int)
    )
    xs = np.unique(
        np.rint(np.linspace(0, nx - 1, SEARCH_GRID_NX)).astype(int)
    )

    if len(ys) != SEARCH_GRID_NY or len(xs) != SEARCH_GRID_NX:
        raise RuntimeError(
            f"无法生成{SEARCH_GRID_NY}×{SEARCH_GRID_NX}唯一索引。"
        )

    pts = [(int(iy), int(ix)) for iy in ys for ix in xs]

    if len(pts) != SEARCH_GRIDPOINT_NUMBER:
        raise RuntimeError(
            f"实际分层点数={len(pts)}，期望={SEARCH_GRIDPOINT_NUMBER}"
        )

    return pts


def read_tyx_fast(
    var,
    meta,
    time_slice,
    y_slice=slice(None),
    x_slice=slice(None),
):
    """和read_tyx相同，但直接返回float32，供矩阵快速VMD使用。"""
    idx = [slice(None)] * 3
    idx[meta["time_axis"]] = time_slice
    idx[meta["lat_axis"]] = y_slice
    idx[meta["lon_axis"]] = x_slice

    arr = var[tuple(idx)]
    arr = np.ma.filled(arr, np.nan)
    arr = np.asarray(arr, dtype=np.float32)
    arr = np.transpose(
        arr,
        axes=[
            meta["time_axis"],
            meta["lat_axis"],
            meta["lon_axis"],
        ],
    )
    return np.ascontiguousarray(arr)


def _read_primary_300_fast(
    var,
    meta,
    train_time_slice,
    ys,
    xs,
    pbar=None,
):
    """
    按时间块一次读取15×20全部空间点。

    旧版是“一个格点读取完整20多年，再读下一个格点”，
    对NetCDF/HDF5非常慢；这里改成“一个时间块同时读取300点”。
    """
    t0 = int(train_time_slice.start)
    t1 = int(train_time_slice.stop)
    ntime = t1 - t0

    matrix = np.empty(
        (SEARCH_GRIDPOINT_NUMBER, ntime),
        dtype=np.float32,
    )

    if pbar is not None:
        pbar.reset(total=ntime)
        pbar.set_description("loading 300 training grids", refresh=True)

    out0 = 0

    for ts in range(t0, t1, int(LOAD_TIME_CHUNK)):
        te = min(ts + int(LOAD_TIME_CHUNK), t1)

        idx = [slice(None)] * 3
        idx[meta["time_axis"]] = slice(ts, te)
        idx[meta["lat_axis"]] = ys
        idx[meta["lon_axis"]] = xs

        block = var[tuple(idx)]
        block = np.ma.filled(block, np.nan)
        block = np.asarray(block, dtype=np.float32)

        block = np.transpose(
            block,
            axes=[
                meta["time_axis"],
                meta["lat_axis"],
                meta["lon_axis"],
            ],
        )

        nt = int(block.shape[0])
        block_300 = block.reshape(nt, -1).T

        if block_300.shape[0] != SEARCH_GRIDPOINT_NUMBER:
            raise RuntimeError(
                f"批量读取得到{block_300.shape[0]}格点，期望300。"
            )

        matrix[:, out0 : out0 + nt] = block_300
        out0 += nt

        if pbar is not None:
            pbar.update(nt)

        del block, block_300

    if out0 != ntime:
        raise RuntimeError(
            f"读取时间长度={out0}，期望={ntime}。"
        )

    return matrix


def collect_300_training_series(
    src_path,
    var_name,
    train_time_slice,
    pbar=None,
):
    """
    高效读取300个空间分层训练格点。

    主路径：15×20规则点一次按时间块批量读取。
    若极少数规则点无效，再逐点寻找替代有效格点。
    """
    with Dataset(src_path, "r") as nc:
        meta = detect_dimensions(nc, var_name)
        var = meta["var"]

        ny = len(nc.dimensions[meta["lat_dim"]])
        nx = len(nc.dimensions[meta["lon_dim"]])

        latv = np.asarray(nc.variables[meta["lat_dim"]][:])
        lonv = np.asarray(nc.variables[meta["lon_dim"]][:])

        primary = stratified_300_indices(ny, nx)
        ys = np.array(sorted({p[0] for p in primary}), dtype=int)
        xs = np.array(sorted({p[1] for p in primary}), dtype=int)

        matrix = _read_primary_300_fast(
            var,
            meta,
            train_time_slice,
            ys,
            xs,
            pbar=pbar,
        )

        records = []
        for iy in ys:
            for ix in xs:
                records.append({
                    "iy": int(iy),
                    "ix": int(ix),
                    "latitude": float(latv[iy]),
                    "longitude": float(lonv[ix]),
                    "selection_source": "15x20_stratified",
                    "series_length": int(matrix.shape[1]),
                })

        finite_mask = np.all(np.isfinite(matrix), axis=1)
        stds = np.nanstd(matrix, axis=1)
        valid_mask = finite_mask & np.isfinite(stds) & (stds > 1e-12)

        # ERA5 SP/T2M/D2M通常全部有效；仅在异常时走替代逻辑。
        if not np.all(valid_mask):
            bad = np.flatnonzero(~valid_mask).tolist()
            print(
                f"[WARNING] 300分层点中有{len(bad)}个无效，"
                "开始从全域补齐。"
            )

            used = {(int(r["iy"]), int(r["ix"])) for r in records}
            replacements = []

            for flat in range(ny * nx):
                iy, ix = divmod(int(flat), nx)
                if (iy, ix) in used:
                    continue

                s = read_point_series(
                    var,
                    meta,
                    train_time_slice,
                    iy,
                    ix,
                )
                s = np.asarray(s, dtype=np.float32)

                if (
                    s.ndim == 1
                    and len(s) == matrix.shape[1]
                    and np.all(np.isfinite(s))
                    and np.nanstd(s) > 1e-12
                ):
                    replacements.append((iy, ix, s))
                    used.add((iy, ix))

                    if len(replacements) >= len(bad):
                        break

            if len(replacements) != len(bad):
                raise RuntimeError(
                    f"无法为{len(bad)}个无效分层点找到足够替代格点。"
                )

            for bad_i, (iy, ix, s) in zip(bad, replacements):
                matrix[bad_i] = s
                records[bad_i] = {
                    "iy": int(iy),
                    "ix": int(ix),
                    "latitude": float(latv[iy]),
                    "longitude": float(lonv[ix]),
                    "selection_source": "replacement_valid_gridpoint",
                    "series_length": int(len(s)),
                }

    return np.ascontiguousarray(matrix, dtype=np.float32), pd.DataFrame(records)


def raw_series_matrix_valid_mask(matrix, eps=1e-12):
    """返回(grid,)有效序列mask。"""
    x = np.asarray(matrix)
    stds = np.nanstd(x, axis=1)
    return (
        np.all(np.isfinite(x), axis=1)
        & np.isfinite(stds)
        & (stds > eps)
    )


def matrix_candidate_scores(
    prepared,
    alpha,
    K,
    batch_size=SEARCH_MATRIX_BATCH,
    pbar=None,
    desc=None,
):
    """
    对prepared中的全部格点计算一个(alpha,K)的格点级objective。

    关键区别：
      旧版 = 300次独立Python VMD
      新版 = 最多300条序列组成一个矩阵，由Numba并行完成VMD

    返回：
      scores: 每格点的mean normalized envelope entropy
      iterations: 每格点实际VMD迭代次数
    """
    n_series = int(prepared["n_series"])
    scores_all = np.full(n_series, np.inf, dtype=np.float64)
    iter_all = np.zeros(n_series, dtype=np.int32)

    if pbar is not None:
        pbar.reset(total=n_series)
        if desc is not None:
            pbar.set_description(desc, refresh=False)
        pbar.refresh()

    for start in range(0, n_series, int(batch_size)):
        stop = min(start + int(batch_size), n_series)
        pb = slice_prepared(prepared, start, stop)

        final_positive, batch_scores, batch_iters = fast_vmd_spectra_and_scores(
            pb,
            alpha=float(alpha),
            K=int(K),
        )

        scores_all[start:stop] = batch_scores
        iter_all[start:stop] = batch_iters

        if pbar is not None:
            pbar.update(stop - start)
            finite_i = batch_iters[batch_iters > 0]
            if finite_i.size:
                pbar.set_postfix_str(
                    f"VMDiter mean={float(np.mean(finite_i)):.1f}, "
                    f"max={int(np.max(finite_i))}",
                    refresh=True,
                )

        del pb, final_positive, batch_scores, batch_iters

    return scores_all, iter_all


def matrix_candidate_objective(
    prepared,
    alpha,
    K,
    batch_size=SEARCH_MATRIX_BATCH,
    pbar=None,
    desc=None,
):
    scores, iters = matrix_candidate_scores(
        prepared,
        alpha=alpha,
        K=K,
        batch_size=batch_size,
        pbar=pbar,
        desc=desc,
    )

    valid = np.isfinite(scores)
    if not np.any(valid):
        return np.inf, {
            "valid_gridpoints": 0,
            "mean_vmd_iterations": np.nan,
            "max_vmd_iterations": 0,
        }

    finite_iters = iters[valid]
    stats = {
        "valid_gridpoints": int(np.sum(valid)),
        "mean_vmd_iterations": float(np.mean(finite_iters)),
        "max_vmd_iterations": int(np.max(finite_iters)),
    }
    return float(np.mean(scores[valid])), stats


def save_pickle_atomic(obj, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")

    with open(tmp, "wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)

    os.replace(tmp, path)


def load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)


In [ ]:
# Cell 5. 固定K下的CFO/GWO/ALA一维alpha搜索
# candidate × grid双矩阵评估；K由Cell 6外层顺序枚举



def _curve_unique_min(x, y):
    """按x排序；重复x仅保留objective最小的一条。"""
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)

    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    if x.size == 0:
        raise ValueError("自动平台选择没有有限候选。")

    order = np.argsort(x)
    x = x[order]
    y = y[order]

    ux = []
    uy = []
    i = 0
    while i < x.size:
        j = i + 1
        while j < x.size and np.isclose(x[j], x[i], rtol=0, atol=1e-10):
            j += 1
        local = y[i:j]
        jj = int(np.nanargmin(local))
        ux.append(float(x[i + jj]))
        uy.append(float(local[jj]))
        i = j

    return np.asarray(ux, dtype=np.float64), np.asarray(uy, dtype=np.float64)


def _normalize01(v, robust_high=False):
    v = np.asarray(v, dtype=np.float64)
    lo = float(np.nanmin(v))

    if robust_high and v.size >= 5:
        hi = float(np.nanpercentile(v, 90.0))
    else:
        hi = float(np.nanmax(v))

    if (not np.isfinite(hi)) or hi <= lo + 1e-15:
        hi = float(np.nanmax(v))

    if hi <= lo + 1e-15:
        return np.zeros_like(v)

    return np.clip((v - lo) / (hi - lo), 0.0, 1.0)


def automatic_low_flat_selection(
    x,
    y,
    prefer_smaller_x=False,
    label="parameter",
):
    """
    无人工百分比阈值的自动“低objective + 平台期”选择。

    情形A：全局最小值在左右边界
      - 对best-so-far单调包络做归一化；
      - 使用首尾弦线最大垂距自动寻找elbow/knee，
        作为继续增大/减小参数收益明显递减的起点。

    情形B：全局最小值位于内部
      - 计算objective和局部|dy/dx|；
      - 选择距离理想点(objective=最低, slope=0)最近的候选；
      - K选择时同时加入较小x的复杂度项，使平台区优先选择更简约K。

    整个过程不需要1%、2%、5%等人工相对改善阈值。
    返回的selected_x一定来自已实际计算过的候选。
    """
    x, y = _curve_unique_min(x, y)

    if x.size == 1:
        return {
            "selected_x": float(x[0]),
            "selected_objective": float(y[0]),
            "raw_min_x": float(x[0]),
            "raw_min_objective": float(y[0]),
            "selection_mode": "single_candidate",
            "selection_score": 0.0,
            "boundary_minimum": True,
            "diagnostics": [
                {
                    "x": float(x[0]),
                    "objective": float(y[0]),
                    "local_abs_slope": 0.0,
                    "automatic_score": 0.0,
                }
            ],
        }

    raw_min_idx = int(np.nanargmin(y))
    boundary_minimum = raw_min_idx in {0, x.size - 1}
    selected_idx = raw_min_idx
    scores = np.full(x.size, np.nan, dtype=np.float64)

    # 局部斜率诊断；np.gradient支持非等间距alpha。
    if x.size >= 3:
        local_slope = np.abs(np.gradient(y, x))
    else:
        slope_value = abs((y[-1] - y[0]) / max(abs(x[-1] - x[0]), 1e-12))
        local_slope = np.full(x.size, slope_value, dtype=np.float64)

    selection_mode = "interior_low_flat_ideal_point"

    if boundary_minimum and x.size >= 3:
        # 将最小值统一放到右端，再做elbow。
        if raw_min_idx == 0:
            xo = -x[::-1]
            yo = y[::-1]
            reverse_back = True
        else:
            xo = x.copy()
            yo = y.copy()
            reverse_back = False

        # best-so-far包络抑制小幅随机反弹。
        y_env = np.minimum.accumulate(yo)

        x_norm = (xo - xo[0]) / max(float(xo[-1] - xo[0]), 1e-12)
        y_span = float(np.nanmax(y_env) - np.nanmin(y_env))

        if y_span <= 1e-15:
            elbow_oriented_idx = len(xo) - 1
            dist = np.zeros_like(xo)
        else:
            y_norm = (y_env - np.nanmin(y_env)) / y_span
            # 连接(0,1)与(1,0)的弦线。
            chord = 1.0 - x_norm
            dist = np.abs(y_norm - chord) / np.sqrt(2.0)

            # 端点本身不能作为真正elbow；仅数值退化时回到raw minimum。
            interior = np.arange(1, len(xo) - 1)
            if interior.size == 0 or float(np.nanmax(dist[interior])) <= 1e-15:
                elbow_oriented_idx = len(xo) - 1
            else:
                elbow_oriented_idx = int(
                    interior[np.nanargmax(dist[interior])]
                )

        # elbow本身是“开始弯折”的位置；为了确保最终点真正位于平台内部，
        # 自动向全局最小值方向再移动一个已实际计算的候选。
        plateau_oriented_idx = min(
            int(elbow_oriented_idx) + 1,
            len(xo) - 1,
        )

        if reverse_back:
            selected_idx = int(x.size - 1 - plateau_oriented_idx)
            scores = dist[::-1].copy()
        else:
            selected_idx = int(plateau_oriented_idx)
            scores = dist.copy()

        # 对elbow模式，较大的弦距代表更强拐点；统一保存为“越小越优”的score。
        finite_max = float(np.nanmax(scores)) if np.any(np.isfinite(scores)) else 0.0
        scores = finite_max - scores
        selection_mode = "boundary_minimum_automatic_elbow"

    else:
        obj_norm = _normalize01(y, robust_high=True)
        slope_norm = _normalize01(local_slope, robust_high=True)

        if prefer_smaller_x:
            x_norm = (x - np.nanmin(x)) / max(
                float(np.nanmax(x) - np.nanmin(x)),
                1e-12,
            )
            scores = np.sqrt(
                (obj_norm**2 + slope_norm**2 + x_norm**2) / 3.0
            )
        else:
            scores = np.sqrt(
                (obj_norm**2 + slope_norm**2) / 2.0
            )

        selected_idx = int(np.nanargmin(scores))

    diagnostics = []
    for i in range(x.size):
        diagnostics.append({
            "x": float(x[i]),
            "objective": float(y[i]),
            "local_abs_slope": float(local_slope[i]),
            "automatic_score": (
                None if not np.isfinite(scores[i]) else float(scores[i])
            ),
        })

    return {
        "selected_x": float(x[selected_idx]),
        "selected_objective": float(y[selected_idx]),
        "raw_min_x": float(x[raw_min_idx]),
        "raw_min_objective": float(y[raw_min_idx]),
        "selection_mode": selection_mode,
        "selection_score": (
            None
            if not np.isfinite(scores[selected_idx])
            else float(scores[selected_idx])
        ),
        "boundary_minimum": bool(boundary_minimum),
        "diagnostics": diagnostics,
        "label": str(label),
    }


def Auto_VMD_300Grid_Search(
    prepared_search,
    optimizer,
    variable_key,
    optimizer_global_index,
    total_optimizer_tasks,
    max_iter=MAX_ITER,
    pop_size=POP_SIZE,
    alpha_bounds=ALPHA_BOUNDS,
    fixed_K=None,
    random_state=None,
    pbar=None,
):
    """
    在给定fixed_K下，只优化连续参数alpha。

    K由Cell 6按K_CANDIDATES顺序枚举；这样既避免把离散K当连续变量搜索，
    也可以直接依据K-objective平台判据停止继续增加模态数。

    evaluate_batch不会消耗随机数，因此同一代candidate仍可集中成矩阵评估。
    """
    eps = 1e-12
    optimizer = str(optimizer).lower()

    if optimizer not in {"cfo", "gwo", "ala"}:
        raise ValueError("optimizer必须为cfo/gwo/ala。")

    alpha_min, alpha_max = map(float, alpha_bounds)

    if fixed_K is None:
        raise ValueError("fixed_K不能为空。")
    fixed_K = int(fixed_K)
    if fixed_K not in K_CANDIDATES:
        raise ValueError(
            f"fixed_K={fixed_K}不在K_CANDIDATES={K_CANDIDATES}中。"
        )

    # 元启发式优化器现在只搜索alpha这一维。
    lb = np.array([alpha_min], dtype=np.float64)
    ub = np.array([alpha_max], dtype=np.float64)
    dim = 1

    checkpoint_path = (
        CHECKPOINT_DIR
        / f"search_{variable_key}_{optimizer}_K{fixed_K}.pkl"
    )

    def sanitize(params):
        alpha = float(np.clip(params[0], alpha_min, alpha_max))
        return alpha, fixed_K, np.array([alpha], dtype=np.float64)

    def clip_position(x):
        return np.clip(x, lb, ub)

    def init_population(rng):
        """
        一维alpha采用分层初始种群：
        - 明确覆盖上下边界；
        - 内部点在等距位置附近小幅随机扰动；
        - 再随机打乱顺序。
        这样POP_SIZE=8仍能完整覆盖0.5–10，而不是偶然只集中在局部。
        """
        n = int(pop_size)
        if n <= 1:
            return np.array([[(alpha_min + alpha_max) / 2.0]], dtype=np.float64)

        values = np.linspace(alpha_min, alpha_max, n, dtype=np.float64)
        if n > 2:
            step = (alpha_max - alpha_min) / float(n - 1)
            jitter = rng.uniform(-0.20, 0.20, size=n - 2) * step
            values[1:-1] = np.clip(
                values[1:-1] + jitter,
                alpha_min,
                alpha_max,
            )

        values[0] = alpha_min
        values[-1] = alpha_max
        rng.shuffle(values)
        return values[:, None]

    # 搜索频谱cache：GPU只上传一次；CPU fallback直接复用prepared_search
    if COMPUTE_DEVICE.startswith("cuda"):
        compute_cache = prepared_to_torch_cache(prepared_search)
    else:
        compute_cache = prepared_search

    objective_cache = {}
    eval_counter = 0
    current_iteration = 0
    batch_time_sum = 0.0
    batch_candidate_count = 0
    last_eval_stats = None

    def evaluate_batch(params_batch, phase_label="population"):
        nonlocal eval_counter, batch_time_sum, batch_candidate_count, last_eval_stats

        params_batch = np.asarray(params_batch, dtype=np.float64)
        if params_batch.ndim == 1:
            params_batch = params_batch[None, :]

        n = params_batch.shape[0]
        results = np.full(n, np.inf, dtype=np.float64)

        # 先查cache，同时对本批内部完全相同(alpha,K)去重。
        uncached_keys = []
        uncached_params = []
        positions_by_key = {}

        for i in range(n):
            alpha, K, _ = sanitize(params_batch[i])
            key = (round(alpha, 8), int(K))

            if key in objective_cache:
                results[i] = float(objective_cache[key])
            else:
                positions_by_key.setdefault(key, []).append(i)
                if len(positions_by_key[key]) == 1:
                    uncached_keys.append(key)
                    uncached_params.append((alpha, K))

        if len(uncached_params) == 0:
            return results

        eval_counter += len(uncached_params)
        desc = (
            f"{variable_key}-K{fixed_K}-{optimizer.upper()} | "
            f"optimizer {optimizer_global_index}/{total_optimizer_tasks} | "
            f"iter {current_iteration}/{int(max_iter)} | "
            f"{phase_label}: {len(uncached_params)} candidates"
        )

        t0 = time.perf_counter()

        if COMPUTE_DEVICE.startswith("cuda"):
            objs, stats = candidate_objectives_population_matrix(
                compute_cache,
                uncached_params,
                pbar=pbar,
                desc=desc,
            )
        else:
            # CPU fallback：仍保持candidate批接口，但逐candidate使用300-grid矩阵。
            objs = []
            iter_values = []
            for jj, (alpha, K) in enumerate(uncached_params):
                obj, stats_one = matrix_candidate_objective(
                    compute_cache,
                    alpha=alpha,
                    K=K,
                    batch_size=SEARCH_MATRIX_BATCH,
                    pbar=pbar,
                    desc=f"{desc} | {jj+1}/{len(uncached_params)}",
                )
                objs.append(obj)
                iter_values.append(stats_one["mean_vmd_iterations"])
            objs = np.asarray(objs, dtype=np.float64)
            stats = {
                "mean_vmd_iterations": float(np.nanmean(iter_values)),
                "max_vmd_iterations": 0,
                "candidate_count": len(uncached_params),
                "task_count": len(uncached_params) * int(prepared_search["n_series"]),
            }

        elapsed = time.perf_counter() - t0
        last_eval_stats = stats
        batch_time_sum += float(elapsed)
        batch_candidate_count += int(len(uncached_params))

        for key, obj in zip(uncached_keys, objs):
            objective_cache[key] = float(obj)
            for pos in positions_by_key[key]:
                results[pos] = float(obj)

        effective_sec = elapsed / max(len(uncached_params), 1)
        avg_effective_sec = batch_time_sum / max(batch_candidate_count, 1)
        projected_days = (
            avg_effective_sec * float(MAX_TOTAL_SEARCH_CANDIDATES) / 86400.0
        )

        if pbar is not None:
            pbar.set_postfix_str(
                f"batch={elapsed:.1f}s/{len(uncached_params)}cand, "
                f"eff={effective_sec:.1f}s/cand, avg={avg_effective_sec:.1f}s/cand, "
                f"proj={projected_days:.2f}d, "
                f"VMDiter={stats.get('mean_vmd_iterations', np.nan):.1f}",
                refresh=True,
            )

        return results

    def evaluate_one(params):
        return float(evaluate_batch(np.asarray(params)[None, :], phase_label="single")[0])

    # checkpoint；旧solver断点不混用
    state = None
    if RESUME_OPTIMIZER_ITERATIONS and checkpoint_path.exists():
        state = load_pickle(checkpoint_path)
        state_alpha_bounds = state.get("alpha_bounds", [np.nan, np.nan])
        valid_state = (
            state.get("optimizer") == optimizer
            and state.get("variable_key") == variable_key
            and int(state.get("fixed_K", -1)) == int(fixed_K)
            and state.get("max_iter") == int(max_iter)
            and state.get("pop_size") == int(pop_size)
            and int(state.get("random_state", -1)) == int(random_state)
            and np.allclose(
                np.asarray(state_alpha_bounds, dtype=np.float64),
                np.asarray(alpha_bounds, dtype=np.float64),
                rtol=0,
                atol=0,
            )
            and bool(state.get("optimization_early_stopping", False))
                == bool(OPTIMIZATION_EARLY_STOPPING)
            and int(state.get("patience", -1)) == int(PATIENCE)
            and np.isclose(
                float(state.get("alpha_relative_improvement_threshold", np.nan)),
                float(ALPHA_REL_IMPROVEMENT_THRESHOLD),
                rtol=0,
                atol=0,
            )
            and state.get("solver_signature") == FAST_SOLVER_SIGNATURE
        )
        if valid_state:
            print(
                f"[RESUME] {variable_key}-K{fixed_K}-{optimizer.upper()} "
                f"from iteration {state['next_iter']}"
            )
        else:
            print(
                f"[WARNING] {checkpoint_path.name}来自旧solver/不同配置，忽略旧断点。"
            )
            state = None

    if state is None:
        rng = np.random.default_rng(random_state)
        X = init_population(rng)
        current_iteration = 0
        fitness = evaluate_batch(X, phase_label="initial population")

        best_idx = int(np.nanargmin(fitness))
        best_score = float(fitness[best_idx])
        best_pos = X[best_idx].copy()
        start_iter = 0
        es_best = np.inf
        es_wait = 0
    else:
        rng = np.random.default_rng()
        rng.bit_generator.state = state["rng_state"]
        X = np.asarray(state["X"], dtype=np.float64)
        fitness = np.asarray(state["fitness"], dtype=np.float64)
        best_score = float(state["best_score"])
        best_pos = np.asarray(state["best_pos"], dtype=np.float64)
        start_iter = int(state["next_iter"])
        current_iteration = start_iter
        es_best = float(state.get("es_best", np.inf))
        es_wait = int(state.get("es_wait", 0))
        objective_cache.update(state.get("objective_cache", {}))
        eval_counter = int(state.get("eval_counter", len(objective_cache)))
        batch_time_sum = float(state.get("batch_time_sum", 0.0))
        batch_candidate_count = int(state.get("batch_candidate_count", 0))

    def save_state(next_iter):
        save_pickle_atomic(
            {
                "solver_signature": FAST_SOLVER_SIGNATURE,
                "optimizer": optimizer,
                "variable_key": variable_key,
                "fixed_K": int(fixed_K),
                "alpha_bounds": [float(alpha_bounds[0]), float(alpha_bounds[1])],
                "max_iter": int(max_iter),
                "pop_size": int(pop_size),
                "random_state": int(random_state),
                "optimization_early_stopping": bool(OPTIMIZATION_EARLY_STOPPING),
                "patience": int(PATIENCE),
                "alpha_relative_improvement_threshold": float(
                    ALPHA_REL_IMPROVEMENT_THRESHOLD
                ),
                "next_iter": int(next_iter),
                "X": X,
                "fitness": fitness,
                "best_score": best_score,
                "best_pos": best_pos,
                "rng_state": rng.bit_generator.state,
                "objective_cache": objective_cache,
                "eval_counter": int(eval_counter),
                "batch_time_sum": float(batch_time_sum),
                "batch_candidate_count": int(batch_candidate_count),
                "es_best": es_best,
                "es_wait": es_wait,
            },
            checkpoint_path,
        )

    # 初始种群完成后立即保存一次checkpoint。
    # 如果程序在第1代过程中断，下次可以直接复用初始种群和objective_cache。
    if state is None and RESUME_OPTIMIZER_ITERATIONS:
        save_state(0)

    def update_early_stopping():
        """
        alpha搜索早停使用“累计相对改善”而不是单纯绝对差。

        只有当当前全局best相对上一次显著改善基准至少下降
        ALPHA_REL_IMPROVEMENT_THRESHOLD时才重置patience。
        """
        nonlocal es_best, es_wait

        if not np.isfinite(es_best):
            es_best = float(best_score)
            es_wait = 0
            return False

        denom = max(abs(float(es_best)), eps)
        relative_improvement = (
            float(es_best) - float(best_score)
        ) / denom

        significant = (
            (float(es_best) - float(best_score)) > float(MIN_DELTA)
            and relative_improvement >= float(ALPHA_REL_IMPROVEMENT_THRESHOLD)
        )

        if significant:
            es_best = float(best_score)
            es_wait = 0
        else:
            es_wait += 1

        return (
            bool(OPTIMIZATION_EARLY_STOPPING)
            and es_wait >= int(PATIENCE)
        )

    # GWO：先按原RNG顺序生成整代candidate，再一次矩阵objective
    if optimizer == "gwo":
        for tt in range(start_iter, int(max_iter)):
            current_iteration = tt + 1
            order = np.argsort(fitness)
            alpha_pos = X[order[0]].copy()
            beta_pos = X[order[1]].copy()
            delta_pos = X[order[2]].copy()

            a = 0.0 if int(max_iter) == 1 else 2.0 - 2.0*tt/(int(max_iter)-1)
            candidates = np.empty_like(X)

            for i in range(int(pop_size)):
                r1 = rng.random(dim); r2 = rng.random(dim)
                A1 = 2*a*r1-a; C1 = 2*r2
                X1 = alpha_pos - A1*np.abs(C1*alpha_pos-X[i])

                r1 = rng.random(dim); r2 = rng.random(dim)
                A2 = 2*a*r1-a; C2 = 2*r2
                X2 = beta_pos - A2*np.abs(C2*beta_pos-X[i])

                r1 = rng.random(dim); r2 = rng.random(dim)
                A3 = 2*a*r1-a; C3 = 2*r2
                X3 = delta_pos - A3*np.abs(C3*delta_pos-X[i])

                candidates[i] = clip_position((X1+X2+X3)/3.0)

            new_fits = evaluate_batch(candidates, phase_label="GWO population")
            finite = np.isfinite(new_fits)
            X[finite] = candidates[finite]
            fitness[finite] = new_fits[finite]

            idx = int(np.nanargmin(fitness))
            if fitness[idx] < best_score:
                best_score = float(fitness[idx])
                best_pos = X[idx].copy()

            stop_now = update_early_stopping()
            save_state(tt + 1)
            if stop_now:
                break

    # ALA
    elif optimizer == "ala":
        vec_flag = np.array([1.0, -1.0])

        def levy(d):
            beta = 1.5
            sigma = (
                gamma(1+beta)*np.sin(np.pi*beta/2)
                / (gamma((1+beta)/2)*beta*2**((beta-1)/2))
            ) ** (1/beta)
            u = rng.standard_normal(d)*sigma
            v = rng.standard_normal(d)
            return u/(np.abs(v)+eps)**(1/beta)

        for tt in range(start_iter, int(max_iter)):
            current_iteration = tt + 1
            iter_num = tt + 1
            RB = rng.standard_normal((int(pop_size), dim))
            F = rng.choice(vec_flag)
            theta = 2.0*np.arctan(1.0 - iter_num/int(max_iter))
            X_new_all = X.copy()

            for i in range(int(pop_size)):
                rand_val = max(rng.random(), eps)
                E = 2.0*np.log(1.0/rand_val)*theta
                if E > 1.0:
                    if rng.random() < 0.3:
                        r1 = 2.0*rng.random(dim)-1.0
                        j = rng.integers(0, int(pop_size))
                        X_new_all[i] = (
                            best_pos + F*RB[i]*(
                                r1*(best_pos-X[i]) + (1-r1)*(X[i]-X[j])
                            )
                        )
                    else:
                        r2 = rng.random()*(1.0+np.sin(0.5*iter_num))
                        j = rng.integers(0, int(pop_size))
                        X_new_all[i] = X[i] + F*r2*(best_pos-X[j])
                else:
                    if rng.random() < 0.5:
                        radius = np.sqrt(np.sum((best_pos-X[i])**2))
                        r3 = rng.random()
                        spiral = radius*(np.sin(2*np.pi*r3)+np.cos(2*np.pi*r3))
                        X_new_all[i] = best_pos + F*X[i]*spiral*rng.random()
                    else:
                        G = 2.0*np.sign(rng.random()-0.5)*(1.0-iter_num/int(max_iter))
                        X_new_all[i] = best_pos + F*G*levy(dim)*(best_pos-X[i])

            candidates = np.vstack([clip_position(x) for x in X_new_all])
            new_fits = evaluate_batch(candidates, phase_label="ALA population")

            for i in range(int(pop_size)):
                if new_fits[i] < fitness[i]:
                    X[i] = candidates[i]
                    fitness[i] = new_fits[i]
                    if new_fits[i] < best_score:
                        best_score = float(new_fits[i])
                        best_pos = candidates[i].copy()

            stop_now = update_early_stopping()
            save_state(tt + 1)
            if stop_now:
                break

    # CFO：exploration整代一起算，exploitation整代一起算
    elif optimizer == "cfo":
        for tt in range(start_iter, int(max_iter)):
            current_iteration = tt + 1
            best_idx = int(np.nanargmin(fitness))
            current_best = X[best_idx].copy()
            sorted_idx = np.argsort(fitness)
            sorted_X = X[sorted_idx].copy()
            sorted_fit = fitness[sorted_idx].copy()
            X_exp = np.zeros_like(X)

            if rng.random() < 0.5:
                r1 = rng.random(); q = rng.integers(1, 3)
                beta = 2.0*np.cos(np.pi*r1)*abs((tt/int(max_iter))**(r1*q))
                X_exp[0] = (
                    current_best - rng.random(dim)*(current_best-sorted_X[0])
                    + beta*(current_best-sorted_X[0])
                )
            else:
                r = rng.random(dim)
                op_alpha = 2.5*r*np.abs(np.cos(np.pi*r))
                X_exp[0] = (
                    sorted_X[0] - rng.random(dim)*(current_best-sorted_X[0])
                    + op_alpha*(current_best-sorted_X[0])
                )

            for i in range(1, int(pop_size)):
                if rng.random() < 0.5:
                    r1 = rng.random(); q = rng.integers(1, 3)
                    beta = 2.0*np.cos(np.pi*r1)*abs((tt/int(max_iter))**(r1*q))
                    X_exp[i] = (
                        current_best - rng.random(dim)*(sorted_X[i-1]-sorted_X[i])
                        + beta*(current_best-sorted_X[i])
                    )
                else:
                    r = rng.random(dim)
                    op_alpha = 2.5*r*np.abs(np.cos(np.pi*r))
                    X_exp[i] = (
                        sorted_X[i] - rng.random(dim)*(sorted_X[i-1]-sorted_X[i])
                        + op_alpha*(current_best-sorted_X[i])
                    )

            exp_candidates = np.vstack([clip_position(x) for x in X_exp])
            exp_fits = evaluate_batch(exp_candidates, phase_label="CFO exploration")

            for i in range(int(pop_size)):
                if exp_fits[i] < sorted_fit[i]:
                    original_idx = sorted_idx[i]
                    X[original_idx] = exp_candidates[i]
                    fitness[original_idx] = exp_fits[i]

            best_idx = int(np.nanargmin(fitness))
            current_best = X[best_idx].copy()
            exploit_candidates = np.empty_like(X)

            for i in range(int(pop_size)):
                if rng.random() < 0.5:
                    e = 3.0*rng.standard_normal(dim)
                    candidate = X[i] + e*(rng.random()*current_best - rng.random()*X[i])
                else:
                    progress = tt/int(max_iter)
                    f = rng.random()*(1.0-progress)**2
                    candidate = current_best + f*(rng.random()*current_best-rng.random()*X[i])
                exploit_candidates[i] = clip_position(candidate)

            exploit_fits = evaluate_batch(exploit_candidates, phase_label="CFO exploitation")
            better = exploit_fits < fitness
            X[better] = exploit_candidates[better]
            fitness[better] = exploit_fits[better]

            idx = int(np.nanargmin(fitness))
            if fitness[idx] < best_score:
                best_score = float(fitness[idx])
                best_pos = X[idx].copy()

            stop_now = update_early_stopping()
            save_state(tt + 1)
            if stop_now:
                break

    # 全部MAX_ITER完成后，基于所有已实际计算过的alpha候选自动选平台alpha。
    # 不直接机械采用optimizer的绝对最小值。
    alpha_trace = sorted(
        [
            (float(key[0]), float(obj))
            for key, obj in objective_cache.items()
            if int(key[1]) == int(fixed_K) and np.isfinite(obj)
        ],
        key=lambda z: z[0],
    )

    if len(alpha_trace) == 0:
        raise RuntimeError(
            f"{variable_key}-K{fixed_K}-{optimizer.upper()}: "
            "没有可用于alpha平台选择的有效objective记录。"
        )

    alpha_values = np.asarray([z[0] for z in alpha_trace], dtype=np.float64)
    alpha_objectives = np.asarray([z[1] for z in alpha_trace], dtype=np.float64)

    alpha_selection = automatic_low_flat_selection(
        alpha_values,
        alpha_objectives,
        prefer_smaller_x=False,
        label=f"{variable_key}-K{fixed_K}-{optimizer.upper()} alpha",
    )

    alpha_opt = float(alpha_selection["selected_x"])
    K_opt = int(fixed_K)
    final_subset_objective = float(alpha_selection["selected_objective"])

    raw_best_idx = int(np.nanargmin(alpha_objectives))
    raw_best_alpha = float(alpha_values[raw_best_idx])
    raw_best_objective = float(alpha_objectives[raw_best_idx])

    mean_effective_candidate_seconds = (
        float(batch_time_sum / batch_candidate_count)
        if batch_candidate_count > 0 else np.nan
    )

    result = {
        "variable": variable_key,
        "optimizer": optimizer.upper(),
        "alpha": float(alpha_opt),
        "K": int(K_opt),
        "search_subset_objective": float(final_subset_objective),
        "search_gridpoint_number": int(prepared_search["n_series"]),
        "alpha_bounds": [float(alpha_bounds[0]), float(alpha_bounds[1])],
        "K_bounds": [int(fixed_K), int(fixed_K)],
        "fixed_K": int(fixed_K),
        "population_size": int(pop_size),
        "maximum_iterations": int(max_iter),
        "iterations_completed": int(current_iteration),
        "optimization_early_stopping": False,
        "stopped_early": False,
        "alpha_selection_method": PLATEAU_SELECTION_METHOD,
        "alpha_selection_mode": str(alpha_selection["selection_mode"]),
        "alpha_selection_score": alpha_selection["selection_score"],
        "alpha_boundary_minimum_before_platform_selection": bool(
            alpha_selection["boundary_minimum"]
        ),
        "raw_minimum_alpha": float(raw_best_alpha),
        "raw_minimum_objective": float(raw_best_objective),
        "alpha_trace": [
            {
                "alpha": float(a),
                "objective": float(o),
            }
            for a, o in alpha_trace
        ],
        "alpha_selection_diagnostics": alpha_selection["diagnostics"],
        "tau": float(TAU),
        "DC": int(DC),
        "init": int(INIT),
        "tol": float(TOL),
        "VMD_max_iter": int(VMD_MAX_ITER),
        "random_state": int(random_state),
        "solver_signature": FAST_SOLVER_SIGNATURE,
        "candidate_evaluations": int(eval_counter),
        "mean_candidate_seconds": mean_effective_candidate_seconds,
        "compute_backend": COMPUTE_DEVICE,
    }

    # 正常完成后默认保留checkpoint。
    if (
        checkpoint_path.exists()
        and not KEEP_COMPLETED_OPTIMIZER_CHECKPOINTS
    ):
        checkpoint_path.unlink()

    if COMPUTE_DEVICE.startswith("cuda"):
        free_torch_cache(compute_cache)

    return result


In [ ]:
# Cell 6. 完整K=2..9扫描：
# 每个固定K下CFO/GWO/ALA完整运行20代，仅优化alpha；
# alpha和K均在全部候选完成后采用无百分比阈值的自动平台选择。


def load_json_if_exists(path):
    path = Path(path)
    if not path.exists():
        return {}
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json_atomic(obj, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)


METHOD_ORDER = {"CFO": 0, "GWO": 1, "ALA": 2}

SEARCH_CONFIG = {
    "solver_signature": FAST_SOLVER_SIGNATURE,
    "alpha_bounds_by_variable": {
        key: [
            float(ALPHA_BOUNDS_BY_VARIABLE[key][0]),
            float(ALPHA_BOUNDS_BY_VARIABLE[key][1]),
        ]
        for key in SOURCE_SPECS
    },
    "K_candidates": [int(k) for k in K_CANDIDATES],
    "population_size": int(POP_SIZE),
    "maximum_iterations": int(MAX_ITER),
    "optimization_early_stopping": False,
    "K_early_stopping": False,
    "plateau_selection_method": PLATEAU_SELECTION_METHOD,
}


def _k_entry_complete(entry):
    return (
        isinstance(entry, dict)
        and all(method in entry for method in ["CFO", "GWO", "ALA"])
    )


def _all_k_complete(variable_result):
    k_scan = variable_result.get("K_scan", {})
    return all(
        _k_entry_complete(k_scan.get(str(int(K)), {}))
        for K in K_CANDIDATES
    )


def analyze_method_k_scan(variable_result, method_key):
    """
    对某一个optimizer独立分析K=2..9的结果。

    每个K下使用Cell 5已经自动选择的“低objective + 平台alpha”。
    K层面再次使用automatic_low_flat_selection：
      - 若最小objective落在K边界，用elbow/knee找收益递减起点；
      - 若最小值在内部，联合objective、局部斜率和较小K复杂度自动选点。

    不使用1%、2%、5%等人工阈值。
    """
    method_key = str(method_key).upper()

    x = []
    y = []
    rows = []

    for K in K_CANDIDATES:
        entry = variable_result["K_scan"][str(int(K))][method_key]
        x.append(float(K))
        y.append(float(entry["search_subset_objective"]))

    selection = automatic_low_flat_selection(
        x,
        y,
        prefer_smaller_x=True,
        label=f"{method_key} K",
    )

    selected_K = int(round(selection["selected_x"]))

    diagnostics_by_k = {
        int(round(d["x"])): d
        for d in selection["diagnostics"]
    }

    for K in K_CANDIDATES:
        r = variable_result["K_scan"][str(int(K))][method_key]
        d = diagnostics_by_k[int(K)]

        rows.append({
            "K": int(K),
            "optimizer": method_key,
            "platform_alpha": float(r["alpha"]),
            "platform_alpha_objective": float(
                r["search_subset_objective"]
            ),
            "raw_minimum_alpha": float(
                r.get("raw_minimum_alpha", r["alpha"])
            ),
            "raw_minimum_objective": float(
                r.get(
                    "raw_minimum_objective",
                    r["search_subset_objective"],
                )
            ),
            "alpha_selection_mode": str(
                r.get("alpha_selection_mode", "")
            ),
            "alpha_boundary_minimum_before_platform_selection": bool(
                r.get(
                    "alpha_boundary_minimum_before_platform_selection",
                    False,
                )
            ),
            "K_local_abs_slope": float(d["local_abs_slope"]),
            "K_automatic_score": (
                None
                if d["automatic_score"] is None
                else float(d["automatic_score"])
            ),
            "is_selected_K_for_method": int(K) == selected_K,
        })

    selected_result = dict(
        variable_result["K_scan"][str(selected_K)][method_key]
    )
    selected_result["K_selection_method"] = PLATEAU_SELECTION_METHOD
    selected_result["K_selection_mode"] = str(
        selection["selection_mode"]
    )
    selected_result["K_selection_score"] = selection["selection_score"]
    selected_result["raw_minimum_K"] = int(
        round(selection["raw_min_x"])
    )
    selected_result["raw_minimum_K_objective"] = float(
        selection["raw_min_objective"]
    )
    selected_result["K_boundary_minimum_before_platform_selection"] = bool(
        selection["boundary_minimum"]
    )

    return rows, selected_result, selection


search_results = (
    load_json_if_exists(SEARCH_RESULT_JSON)
    if REUSE_FINISHED_SEARCH
    else {}
)

# 变量级搜索配置迁移
# 旧版JSON只有全局alpha_bounds=(0.5,10)。当前只改变T2M的alpha下限，
# 因此不能因为整体_config不同就清空SP。这里先比较除alpha范围外的核心配置；
# 核心配置一致时，再逐变量检查已缓存结果实际记录的alpha_bounds：
#   SP  : 0.5~10 -> 保留；
#   T2M : 0.5~10 -> 目标0.05~10，整变量清除并重新跑K=2..9；
#   D2M : 若已有0.5~10结果则保留，否则正常继续。
def _search_config_without_alpha(config):
    config = dict(config) if isinstance(config, dict) else {}
    config.pop("alpha_bounds", None)
    config.pop("alpha_bounds_by_variable", None)
    return config


def _cached_variable_bounds_match(variable_result, desired_bounds):
    if not isinstance(variable_result, dict):
        return True

    desired = np.asarray(desired_bounds, dtype=np.float64)
    found = []

    for K in K_CANDIDATES:
        k_entry = variable_result.get("K_scan", {}).get(str(int(K)), {})
        if not isinstance(k_entry, dict):
            continue

        for method_key in ["CFO", "GWO", "ALA"]:
            result = k_entry.get(method_key)
            if not isinstance(result, dict) or "alpha_bounds" not in result:
                continue
            found.append(
                np.asarray(result["alpha_bounds"], dtype=np.float64)
            )

    # 没有已经完成的optimizer结果，不构成范围冲突。
    if len(found) == 0:
        return True

    return all(
        arr.shape == desired.shape
        and np.allclose(arr, desired, rtol=0.0, atol=0.0)
        for arr in found
    )


old_config = search_results.get("_config", {}) if search_results else {}

if search_results and old_config != SEARCH_CONFIG:
    old_core = _search_config_without_alpha(old_config)
    new_core = _search_config_without_alpha(SEARCH_CONFIG)

    if old_core == new_core:
        print(
            "[INFO] K/POP/ITER/solver等核心搜索配置未变化；"
            "仅按变量检查alpha范围并选择性复用缓存。"
        )

        for variable_key in SOURCE_SPECS:
            if variable_key not in search_results:
                continue

            desired_bounds = ALPHA_BOUNDS_BY_VARIABLE[variable_key]

            if _cached_variable_bounds_match(
                search_results[variable_key],
                desired_bounds,
            ):
                print(
                    f"[KEEP] {variable_key}: 已有搜索结果与当前alpha范围"
                    f"{tuple(desired_bounds)}兼容，继续复用。"
                )
            else:
                print(
                    f"[RESET] {variable_key}: 已有搜索结果的alpha范围与当前"
                    f"{tuple(desired_bounds)}不一致；仅清除{variable_key}搜索缓存。"
                )
                search_results.pop(variable_key, None)

        search_results["_config"] = SEARCH_CONFIG
        save_json_atomic(search_results, SEARCH_RESULT_JSON)

    else:
        print(
            "[WARNING] K / POP / ITER / solver等核心配置发生变化，"
            "无法安全复用旧搜索结果；清空全部搜索记录。"
        )
        search_results = {"_config": SEARCH_CONFIG}
else:
    search_results["_config"] = SEARCH_CONFIG

search_point_frames = []
train_time_slice = predictor_slices["train"]

total_optimizer_tasks = (
    len(SOURCE_SPECS)
    * len(K_CANDIDATES)
    * len(METHODS)
)
optimizer_global_index = 0

pbar = tqdm(
    total=SEARCH_GRIDPOINT_NUMBER,
    desc="VMD full K/alpha search initialization",
    unit="grid",
    dynamic_ncols=True,
    leave=True,
)

try:
    for variable_key, spec in SOURCE_SPECS.items():
        src_path = SOURCE_DIR / spec["filename"]
        variable_alpha_bounds = ALPHA_BOUNDS_BY_VARIABLE[variable_key]

        # 300点批量读取
        pbar.set_description(
            f"{variable_key} | loading 300 grids",
            refresh=True,
        )

        series_matrix, point_df = collect_300_training_series(
            src_path=src_path,
            var_name=spec["var_name"],
            train_time_slice=train_time_slice,
            pbar=pbar,
        )

        point_df.insert(0, "variable", variable_key)
        search_point_frames.append(point_df)

        variable_result = search_results.setdefault(
            variable_key,
            {"K_scan": {}},
        )
        variable_result.setdefault("K_scan", {})

        all_cached = _all_k_complete(variable_result)

        if all_cached:
            print(
                f"[CACHED VARIABLE] {variable_key}: "
                "K=2..9 × CFO/GWO/ALA均已完整计算；"
                "跳过FFT和VMD搜索，仅重新执行自动平台选择。"
            )
            # 进度编号仍计入已经缓存的24个optimizer任务，使T2M从25/72开始显示。
            optimizer_global_index += (
                len(K_CANDIDATES) * len(METHODS)
            )
            prepared_search = None
        else:
            pbar.reset(total=1)
            pbar.set_description(
                f"{variable_key} | precompute mirror+FFT for 300 grids",
                refresh=True,
            )

            tprep = time.perf_counter()
            prepared_search = prepare_vmd_spectra_matrix(
                series_matrix,
                fft_workers=FFT_WORKERS,
            )
            prep_seconds = time.perf_counter() - tprep
            pbar.update(1)
            pbar.set_postfix_str(
                f"FFT cache={prep_seconds:.1f}s",
                refresh=True,
            )

        del series_matrix
        gc.collect()

        # 所有K必须完整跑完；没有K early stop。
        if not all_cached:
            for K in K_CANDIDATES:
                K = int(K)
                k_key = str(K)
                k_entry = variable_result["K_scan"].setdefault(
                    k_key,
                    {},
                )

                print("\n" + "-" * 100)
                print(
                    f"{variable_key}: fixed K={K}; "
                    f"alpha∈[{variable_alpha_bounds[0]}, {variable_alpha_bounds[1]}]; "
                    f"POP={POP_SIZE}, ITER={MAX_ITER}; no early stopping"
                )
                print("-" * 100)

                for method in METHODS:
                    optimizer_global_index += 1
                    method_key = method.upper()

                    if (
                        REUSE_FINISHED_SEARCH
                        and method_key in k_entry
                    ):
                        cached = k_entry[method_key]
                        pbar.reset(total=SEARCH_GRIDPOINT_NUMBER)
                        pbar.n = SEARCH_GRIDPOINT_NUMBER
                        pbar.set_description(
                            f"{variable_key}-K{K}-{method_key} | CACHED",
                            refresh=False,
                        )
                        pbar.set_postfix_str(
                            f"a(platform)={cached['alpha']:.4f}, "
                            f"obj={cached['search_subset_objective']:.7f}",
                            refresh=True,
                        )
                        continue

                    result = Auto_VMD_300Grid_Search(
                        prepared_search=prepared_search,
                        optimizer=method,
                        variable_key=variable_key,
                        optimizer_global_index=optimizer_global_index,
                        total_optimizer_tasks=total_optimizer_tasks,
                        max_iter=MAX_ITER,
                        pop_size=POP_SIZE,
                        alpha_bounds=variable_alpha_bounds,
                        fixed_K=K,
                        random_state=RANDOM_STATES[method],
                        pbar=pbar,
                    )

                    k_entry[method_key] = result
                    save_json_atomic(
                        search_results,
                        SEARCH_RESULT_JSON,
                    )

                    pbar.reset(total=SEARCH_GRIDPOINT_NUMBER)
                    pbar.n = SEARCH_GRIDPOINT_NUMBER
                    pbar.set_description(
                        f"{variable_key}-K{K}-{method_key} | DONE",
                        refresh=False,
                    )
                    pbar.set_postfix_str(
                        f"a={result['alpha']:.4f}, "
                        f"obj={result['search_subset_objective']:.7f}, "
                        f"iters={result['iterations_completed']}",
                        refresh=True,
                    )

                    print(
                        f"\n[DONE] {variable_key}-K{K}-{method_key}: "
                        f"platform alpha={result['alpha']:.6f}, "
                        f"platform objective={result['search_subset_objective']:.8f}, "
                        f"raw minimum alpha={result['raw_minimum_alpha']:.6f}, "
                        f"raw minimum objective={result['raw_minimum_objective']:.8f}, "
                        f"optimizer iterations={result['iterations_completed']}, "
                        f"candidate evals={result['candidate_evaluations']}, "
                        f"selection={result['alpha_selection_mode']}"
                    )
                    gc.collect()

                save_json_atomic(
                    search_results,
                    SEARCH_RESULT_JSON,
                )

        if not _all_k_complete(variable_result):
            raise RuntimeError(
                f"{variable_key}: K=2..9尚未全部完成，不能进行最终自动平台选择。"
            )

        # 每种optimizer独立选择自己的K平台点。
        # 因而CFO/GWO/ALA最终K可以相同，也可以不同。
        variable_result["method_K_summary"] = {}
        variable_result["method_platform_selection"] = {}

        for method_key in ["CFO", "GWO", "ALA"]:
            k_rows, selected_result, k_selection = analyze_method_k_scan(
                variable_result,
                method_key,
            )

            variable_result["method_K_summary"][method_key] = k_rows
            variable_result["method_platform_selection"][method_key] = {
                "selected_K": int(selected_result["K"]),
                "selected_alpha": float(selected_result["alpha"]),
                "selected_objective": float(
                    selected_result["search_subset_objective"]
                ),
                "selection_mode": str(k_selection["selection_mode"]),
                "selection_score": k_selection["selection_score"],
                "raw_minimum_K": int(
                    round(k_selection["raw_min_x"])
                ),
                "raw_minimum_K_objective": float(
                    k_selection["raw_min_objective"]
                ),
                "boundary_minimum_before_platform_selection": bool(
                    k_selection["boundary_minimum"]
                ),
            }

            # Cell 7读取这里的三种方法最终候选。
            variable_result[method_key] = selected_result

        variable_result["K_selection_reason"] = (
            "CFO/GWO/ALA分别在完整K=2..9扫描后，使用无人工百分比阈值的"
            f"{PLATEAU_SELECTION_METHOD}自动选择其低objective平台K。"
        )

        save_json_atomic(
            search_results,
            SEARCH_RESULT_JSON,
        )

        print("\n" + "=" * 100)
        print(f"{variable_key}: method-specific automatic plateau selection")
        print("=" * 100)
        for method_key in ["CFO", "GWO", "ALA"]:
            r = variable_result[method_key]
            print(
                f"  {method_key}: "
                f"K={r['K']}, alpha={r['alpha']:.6f}, "
                f"objective={r['search_subset_objective']:.8f}, "
                f"K-selection={r['K_selection_mode']}, "
                f"alpha-selection={r['alpha_selection_mode']}"
            )

        if prepared_search is not None:
            del prepared_search
        del point_df
        gc.collect()

finally:
    pbar.close()

search_points_df = pd.concat(
    search_point_frames,
    ignore_index=True,
)

print("\n完整K=2..9 / alpha搜索完成或已恢复。")
print("Search result JSON:", SEARCH_RESULT_JSON)

for variable_key in SOURCE_SPECS:
    vr = search_results[variable_key]

    print("\n" + "=" * 100)
    print(variable_key)
    print("=" * 100)

    for method_key in ["CFO", "GWO", "ALA"]:
        r = vr[method_key]
        raw_K = r.get("raw_minimum_K", r["K"])
        raw_a = r.get("raw_minimum_alpha", r["alpha"])

        print(
            f"{method_key}: "
            f"selected platform K={r['K']}, alpha={r['alpha']:.6f}, "
            f"objective={r['search_subset_objective']:.8f}; "
            f"raw-min K={raw_K}, raw-min alpha={raw_a:.6f}"
        )


In [ ]:
# Cell 7. 三种optimizer各自平台候选：全部训练格点full-domain复核
#
# 最终组合规则：
# 1) 三种方法selected K全部相同 -> 三方法按full-domain objective反比加权；
# 2) 仅两种方法selected K相同 -> 这两种方法按full-domain objective反比加权；
# 3) 三种方法selected K均不同 -> 选择full-domain objective最低的单方法。
#
# 只有K相同的结果才允许逐IMF加权平均。


def full_domain_reevaluate_candidates(
    src_path,
    var_name,
    variable_key,
    train_time_slice,
    optimizer_candidate_results,
):
    optimizer_records = []

    for method in ["CFO", "GWO", "ALA"]:
        r = optimizer_candidate_results[method]

        optimizer_records.append({
            "optimizer": method,
            "alpha": float(r["alpha"]),
            "K": int(r["K"]),
            "search_subset_objective": float(
                r["search_subset_objective"]
            ),
            "alpha_selection_mode": str(
                r.get("alpha_selection_mode", "")
            ),
            "K_selection_mode": str(
                r.get("K_selection_mode", "")
            ),
        })

    # 不同method可能碰巧得到完全相同(alpha,K)，full-domain只算一次。
    unique_map = {}
    unique_candidates = []

    for r in optimizer_records:
        key = (
            round(float(r["alpha"]), 8),
            int(r["K"]),
        )

        if key not in unique_map:
            unique_map[key] = len(unique_candidates)
            unique_candidates.append({
                "alpha": float(r["alpha"]),
                "K": int(r["K"]),
                "key": key,
            })

        r["unique_candidate_index"] = unique_map[key]

    checkpoint_path = (
        CHECKPOINT_DIR
        / f"fullgrid_autoplateau_{variable_key}.pkl"
    )

    if checkpoint_path.exists():
        try:
            old = load_pickle(checkpoint_path)
            if old.get("solver_signature") != FAST_SOLVER_SIGNATURE:
                checkpoint_path.unlink()
        except Exception:
            pass

    with Dataset(src_path, "r") as nc:
        meta = detect_dimensions(nc, var_name)
        var = meta["var"]
        ny = len(nc.dimensions[meta["lat_dim"]])
        nx = len(nc.dimensions[meta["lon_dim"]])

        print(
            f"\n[{variable_key}] reading full {ny}×{nx} "
            "training domain into RAM..."
        )

        cube = read_tyx_fast(
            var,
            meta,
            time_slice=train_time_slice,
            y_slice=slice(0, ny),
            x_slice=slice(0, nx),
        )

    nt = cube.shape[0]

    matrix_all = np.ascontiguousarray(
        np.transpose(cube, (1, 2, 0)).reshape(
            ny * nx,
            nt,
        ),
        dtype=np.float32,
    )

    del cube

    valid_mask = raw_series_matrix_valid_mask(
        matrix_all
    )
    valid_idx = np.flatnonzero(valid_mask)
    raw_valid_count = int(valid_idx.size)

    if raw_valid_count == 0:
        raise RuntimeError(
            f"{variable_key}: full-domain没有有效训练格点。"
        )

    valid_matrix = np.ascontiguousarray(
        matrix_all[valid_idx]
    )
    del matrix_all
    gc.collect()

    print(
        f"[{variable_key}] valid full-domain grids="
        f"{raw_valid_count}; raw matrix="
        f"{valid_matrix.nbytes/1024**3:.2f} GiB"
    )

    t0 = time.perf_counter()

    prepared = prepare_vmd_spectra_matrix(
        valid_matrix,
        fft_workers=FFT_WORKERS,
    )

    del valid_matrix
    gc.collect()

    print(
        f"[{variable_key}] full-domain mirror+FFT cache: "
        f"{time.perf_counter()-t0:.1f}s, "
        f"f_hat={prepared['f_hat_pos'].nbytes/1024**3:.2f} GiB"
    )

    params = [
        (c["alpha"], c["K"])
        for c in unique_candidates
    ]

    pbar = tqdm(
        total=len(params) * raw_valid_count,
        desc=f"{variable_key} full-domain candidate×grid",
        unit="task",
        dynamic_ncols=True,
    )

    try:
        if COMPUTE_DEVICE.startswith("cuda"):
            gpu_cache = prepared_to_torch_cache(
                prepared
            )

            objectives, stats = (
                candidate_objectives_population_matrix(
                    gpu_cache,
                    params,
                    pbar=pbar,
                    desc=(
                        f"{variable_key} full-domain "
                        f"{len(params)} candidates"
                    ),
                )
            )

            free_torch_cache(
                gpu_cache
            )

        else:
            objectives = []

            for alpha, K in params:
                obj, _ = matrix_candidate_objective(
                    prepared,
                    alpha=alpha,
                    K=K,
                    batch_size=SEARCH_MATRIX_BATCH,
                    pbar=None,
                )

                objectives.append(
                    obj
                )
                pbar.update(
                    raw_valid_count
                )

            objectives = np.asarray(
                objectives,
                dtype=np.float64,
            )

    finally:
        pbar.close()

    del prepared
    gc.collect()

    records = []

    for r in optimizer_records:
        ic = r["unique_candidate_index"]

        records.append({
            "variable": variable_key,
            "optimizer": r["optimizer"],
            "alpha": r["alpha"],
            "K": r["K"],
            "search_subset_objective": r[
                "search_subset_objective"
            ],
            "full_domain_objective": float(
                objectives[ic]
            ),
            "full_domain_valid_gridpoint_number": (
                raw_valid_count
            ),
            "raw_valid_training_gridpoint_number": (
                raw_valid_count
            ),
            "alpha_selection_mode": r[
                "alpha_selection_mode"
            ],
            "K_selection_mode": r[
                "K_selection_mode"
            ],
        })

    if checkpoint_path.exists():
        checkpoint_path.unlink()

    return records


def _inverse_objective_weights(records):
    """
    objective越低权重越高：
        w_i = (1/J_i) / sum_j(1/J_j)

    normalized envelope entropy为非负量；使用eps防止数值除零。
    """
    objs = np.asarray(
        [
            max(
                abs(float(r["full_domain_objective"])),
                1e-12,
            )
            for r in records
        ],
        dtype=np.float64,
    )

    inv = 1.0 / objs
    weights = inv / np.sum(inv)

    return weights


def _build_final_selection(
    variable_key,
    records,
):
    """
    根据K一致性生成最终单方法或加权ensemble配置。
    """
    if len(records) != 3:
        raise ValueError(
            f"{variable_key}: 必须有CFO/GWO/ALA三条full-domain记录。"
        )

    k_counts = {}
    for r in records:
        K = int(r["K"])
        k_counts[K] = k_counts.get(K, 0) + 1

    modal_K = None
    modal_count = max(k_counts.values())

    if modal_count >= 2:
        modal_K = min(
            K
            for K, count in k_counts.items()
            if count == modal_count
        )

        members = [
            dict(r)
            for r in records
            if int(r["K"]) == int(modal_K)
        ]

        weights = _inverse_objective_weights(
            members
        )

        for r, w in zip(members, weights):
            r["weight"] = float(w)

        representative_alpha = float(
            np.sum(
                [
                    float(r["alpha"]) * float(r["weight"])
                    for r in members
                ]
            )
        )

        weighted_reference_objective = float(
            np.sum(
                [
                    float(r["full_domain_objective"])
                    * float(r["weight"])
                    for r in members
                ]
            )
        )

        selection = {
            "variable": variable_key,
            "selection_mode": "weighted_ensemble",
            "optimizer": "+".join(
                r["optimizer"]
                for r in members
            ),
            "K": int(modal_K),
            # 仅作为汇总字段；真正分解使用members中的各自alpha。
            "alpha": representative_alpha,
            "representative_alpha": representative_alpha,
            "member_count": int(len(members)),
            "members": members,
            "search_subset_objective": float(
                np.sum(
                    [
                        float(r["search_subset_objective"])
                        * float(r["weight"])
                        for r in members
                    ]
                )
            ),
            "full_domain_objective": (
                weighted_reference_objective
            ),
            "full_domain_objective_is_member_weighted_reference": True,
            "full_domain_valid_gridpoint_number": int(
                members[0][
                    "full_domain_valid_gridpoint_number"
                ]
            ),
            "raw_valid_training_gridpoint_number": int(
                members[0][
                    "raw_valid_training_gridpoint_number"
                ]
            ),
            "selection_reason": (
                f"{len(members)}个optimizer自动平台选择得到相同K="
                f"{modal_K}；对应VMD结果使用full-domain "
                "objective反比权重逐IMF加权平均。"
            ),
        }

        return selection

    # 三个K完全不同：三个候选本身均已通过自动平台选择，
    # 再按full-domain objective选主方法。
    method_order = {
        "CFO": 0,
        "GWO": 1,
        "ALA": 2,
    }

    winner = min(
        records,
        key=lambda r: (
            float(r["full_domain_objective"]),
            float(r["search_subset_objective"]),
            method_order[r["optimizer"]],
        ),
    )

    member = dict(winner)
    member["weight"] = 1.0

    return {
        "variable": variable_key,
        "selection_mode": "single_method",
        "optimizer": winner["optimizer"],
        "K": int(winner["K"]),
        "alpha": float(winner["alpha"]),
        "representative_alpha": float(
            winner["alpha"]
        ),
        "member_count": 1,
        "members": [member],
        "search_subset_objective": float(
            winner["search_subset_objective"]
        ),
        "full_domain_objective": float(
            winner["full_domain_objective"]
        ),
        "full_domain_objective_is_member_weighted_reference": False,
        "full_domain_valid_gridpoint_number": int(
            winner[
                "full_domain_valid_gridpoint_number"
            ]
        ),
        "raw_valid_training_gridpoint_number": int(
            winner[
                "raw_valid_training_gridpoint_number"
            ]
        ),
        "selection_reason": (
            "CFO/GWO/ALA自动平台选择得到三个不同K；"
            f"选择full-domain objective最低的{winner['optimizer']} "
            f"(K={winner['K']}, alpha={winner['alpha']:.6f})作为主方法。"
        ),
    }


full_reeval_results = (
    load_json_if_exists(FULL_REEVAL_JSON)
    if REUSE_FINISHED_FULL_REEVALUATION
    else {}
)


def full_reeval_cache_matches(
    variable_key,
):
    if variable_key not in full_reeval_results:
        return False

    records = full_reeval_results[
        variable_key
    ]

    if (
        not isinstance(records, list)
        or len(records) != 3
    ):
        return False

    by_method = {
        str(r.get("optimizer")): r
        for r in records
    }

    for method in [
        "CFO",
        "GWO",
        "ALA",
    ]:
        if method not in by_method:
            return False

        expected = search_results[
            variable_key
        ][method]

        cached = by_method[
            method
        ]

        if int(
            cached.get(
                "K",
                -1,
            )
        ) != int(
            expected["K"]
        ):
            return False

        if not np.isclose(
            float(
                cached.get(
                    "alpha",
                    np.nan,
                )
            ),
            float(
                expected["alpha"]
            ),
            rtol=0,
            atol=1e-10,
        ):
            return False

    return True


for variable_key, spec in SOURCE_SPECS.items():
    if (
        REUSE_FINISHED_FULL_REEVALUATION
        and full_reeval_cache_matches(
            variable_key
        )
    ):
        print(
            f"[SKIP] {variable_key} 已有与当前三个自动平台候选"
            "完全匹配的full-domain复核结果。"
        )
        continue

    records = full_domain_reevaluate_candidates(
        src_path=SOURCE_DIR / spec["filename"],
        var_name=spec["var_name"],
        variable_key=variable_key,
        train_time_slice=predictor_slices[
            "train"
        ],
        optimizer_candidate_results=search_results[
            variable_key
        ],
    )

    full_reeval_results[
        variable_key
    ] = records

    save_json_atomic(
        full_reeval_results,
        FULL_REEVAL_JSON,
    )

    gc.collect()


selected_params = {}
all_full_records = []

for variable_key in SOURCE_SPECS:
    records = full_reeval_results[
        variable_key
    ]

    all_full_records.extend(
        records
    )

    selected_params[
        variable_key
    ] = _build_final_selection(
        variable_key,
        records,
    )

    # 把最终K/方式同步回搜索JSON，便于后续恢复和审计。
    search_results[variable_key][
        "selected_K"
    ] = int(
        selected_params[
            variable_key
        ]["K"]
    )

    search_results[variable_key][
        "final_selection_mode"
    ] = str(
        selected_params[
            variable_key
        ]["selection_mode"]
    )

    search_results[variable_key][
        "final_selection_reason"
    ] = str(
        selected_params[
            variable_key
        ]["selection_reason"]
    )

save_json_atomic(
    search_results,
    SEARCH_RESULT_JSON,
)

save_json_atomic(
    selected_params,
    FINAL_SELECTION_JSON,
)

full_reeval_df = pd.DataFrame(
    all_full_records
)

selected_rows = []

for variable_key, p in selected_params.items():
    member_text = "; ".join(
        [
            (
                f"{m['optimizer']}:K={m['K']},"
                f"alpha={m['alpha']:.6f},"
                f"w={m['weight']:.6f},"
                f"Jfull={m['full_domain_objective']:.8f}"
            )
            for m in p["members"]
        ]
    )

    selected_rows.append({
        "variable": variable_key,
        "selection_mode": p[
            "selection_mode"
        ],
        "selected_optimizer_or_ensemble": p[
            "optimizer"
        ],
        "selected_K": p["K"],
        "representative_alpha": p[
            "representative_alpha"
        ],
        "member_count": p[
            "member_count"
        ],
        "members": member_text,
        "member_weighted_reference_objective": p[
            "full_domain_objective"
        ],
        "selection_reason": p[
            "selection_reason"
        ],
        "full_domain_valid_gridpoint_number": p[
            "full_domain_valid_gridpoint_number"
        ],
    })

selected_df = pd.DataFrame(
    selected_rows
)

print("\n" + "=" * 110)
print(
    "最终变量特异性VMD配置："
    "三方法K多数一致则同K加权ensemble；K全不同则选最低full-domain objective单方法。"
)
print("=" * 110)

display(
    selected_df
)

print(
    "Final selection JSON:",
    FINAL_SELECTION_JSON,
)


In [ ]:
# Cell 8. 保存完整K/alpha搜索、自动平台选择、full-domain复核和最终ensemble信息

# A. 所有K × optimizer搜索结果
all_k_search_rows = []
method_k_summary_rows = []

for variable_key in SOURCE_SPECS:
    vr = search_results[variable_key]

    for K in K_CANDIDATES:
        k_entry = vr.get(
            "K_scan",
            {},
        ).get(
            str(int(K))
        )

        if not _k_entry_complete(
            k_entry
        ):
            continue

        for method in [
            "CFO",
            "GWO",
            "ALA",
        ]:
            r = k_entry[
                method
            ]

            all_k_search_rows.append({
                "variable": variable_key,
                "K": int(K),
                "optimizer": method,
                "platform_alpha": float(
                    r["alpha"]
                ),
                "platform_alpha_objective": float(
                    r[
                        "search_subset_objective"
                    ]
                ),
                "raw_minimum_alpha": float(
                    r.get(
                        "raw_minimum_alpha",
                        r["alpha"],
                    )
                ),
                "raw_minimum_objective": float(
                    r.get(
                        "raw_minimum_objective",
                        r[
                            "search_subset_objective"
                        ],
                    )
                ),
                "alpha_selection_mode": str(
                    r.get(
                        "alpha_selection_mode",
                        "",
                    )
                ),
                "alpha_selection_score": r.get(
                    "alpha_selection_score",
                    None,
                ),
                "alpha_boundary_minimum_before_platform_selection": bool(
                    r.get(
                        "alpha_boundary_minimum_before_platform_selection",
                        False,
                    )
                ),
                "search_gridpoint_number": int(
                    r[
                        "search_gridpoint_number"
                    ]
                ),
                "alpha_min": float(
                    r["alpha_bounds"][0]
                ),
                "alpha_max": float(
                    r["alpha_bounds"][1]
                ),
                "population_size": int(
                    r["population_size"]
                ),
                "maximum_iterations": int(
                    r[
                        "maximum_iterations"
                    ]
                ),
                "iterations_completed": int(
                    r.get(
                        "iterations_completed",
                        r[
                            "maximum_iterations"
                        ],
                    )
                ),
                "candidate_evaluations": int(
                    r.get(
                        "candidate_evaluations",
                        -1,
                    )
                ),
                "mean_candidate_seconds": float(
                    r.get(
                        "mean_candidate_seconds",
                        np.nan,
                    )
                ),
                "tau": float(
                    r["tau"]
                ),
                "DC": int(
                    r["DC"]
                ),
                "init": int(
                    r["init"]
                ),
                "tol": float(
                    r["tol"]
                ),
                "VMD_max_iter": int(
                    r[
                        "VMD_max_iter"
                    ]
                ),
                "random_state": int(
                    r[
                        "random_state"
                    ]
                ),
            })

    for method in [
        "CFO",
        "GWO",
        "ALA",
    ]:
        for row in vr.get(
            "method_K_summary",
            {},
        ).get(
            method,
            [],
        ):
            rr = dict(row)
            rr[
                "variable"
            ] = variable_key
            method_k_summary_rows.append(
                rr
            )


all_k_search_df = pd.DataFrame(
    all_k_search_rows
)

method_k_summary_df = pd.DataFrame(
    method_k_summary_rows
)


# B. 每种optimizer最终自动平台候选
method_selected_rows = []

for variable_key in SOURCE_SPECS:
    vr = search_results[
        variable_key
    ]

    for method in [
        "CFO",
        "GWO",
        "ALA",
    ]:
        r = vr[
            method
        ]

        method_selected_rows.append({
            "variable": variable_key,
            "optimizer": method,
            "selected_platform_K": int(
                r["K"]
            ),
            "selected_platform_alpha": float(
                r["alpha"]
            ),
            "search_subset_objective": float(
                r[
                    "search_subset_objective"
                ]
            ),
            "K_selection_mode": str(
                r.get(
                    "K_selection_mode",
                    "",
                )
            ),
            "K_selection_score": r.get(
                "K_selection_score",
                None,
            ),
            "raw_minimum_K": int(
                r.get(
                    "raw_minimum_K",
                    r["K"],
                )
            ),
            "raw_minimum_K_objective": float(
                r.get(
                    "raw_minimum_K_objective",
                    r[
                        "search_subset_objective"
                    ],
                )
            ),
            "alpha_selection_mode": str(
                r.get(
                    "alpha_selection_mode",
                    "",
                )
            ),
            "raw_minimum_alpha": float(
                r.get(
                    "raw_minimum_alpha",
                    r["alpha"],
                )
            ),
            "raw_minimum_alpha_objective": float(
                r.get(
                    "raw_minimum_objective",
                    r[
                        "search_subset_objective"
                    ],
                )
            ),
        })


method_selected_df = pd.DataFrame(
    method_selected_rows
)


# C. Table S4 / 论文方法汇总
paper_rows = []

for variable_key in SOURCE_SPECS:
    p = selected_params[
        variable_key
    ]

    member_text = "; ".join(
        [
            (
                f"{m['optimizer']}: "
                f"K={m['K']}, "
                f"alpha={m['alpha']:.6f}, "
                f"weight={m['weight']:.6f}"
            )
            for m in p[
                "members"
            ]
        ]
    )

    paper_rows.append({
        "Predictor": variable_key,
        "Search gridpoint number": (
            SEARCH_GRIDPOINT_NUMBER
        ),
        "Search spatial sampling": (
            "15 × 20 spatially stratified "
            "approximately uniform valid training gridpoints"
        ),
        "Full-domain re-evaluation": (
            "All valid training-period gridpoints"
        ),
        "Candidate optimizers": (
            "CFO, GWO, ALA"
        ),
        "Objective": (
            "Mean normalized envelope entropy"
        ),
        "alpha search interval": (
            f"{ALPHA_BOUNDS_BY_VARIABLE[variable_key][0]}–"
            f"{ALPHA_BOUNDS_BY_VARIABLE[variable_key][1]}"
        ),
        "K candidate set": ", ".join(
            str(
                int(k)
            )
            for k in K_CANDIDATES
        ),
        "K search strategy": (
            "Complete fixed-K scan; alpha optimized "
            "independently at every K"
        ),
        "Automatic alpha/K platform selection": (
            PLATEAU_SELECTION_METHOD
        ),
        "Artificial percentage threshold": (
            "None"
        ),
        "Population size": (
            POP_SIZE
        ),
        "Maximum optimizer iterations": (
            MAX_ITER
        ),
        "Optimizer early stopping": (
            "Disabled; every optimizer completes all iterations"
        ),
        "tau": (
            TAU
        ),
        "DC": (
            DC
        ),
        "Initialization mode": (
            INIT
        ),
        "Convergence tolerance": (
            TOL
        ),
        "Odd-length handling": (
            "Repeat final value and truncate after decomposition"
        ),
        "Final selection mode": (
            p[
                "selection_mode"
            ]
        ),
        "Final optimizer/ensemble": (
            p[
                "optimizer"
            ]
        ),
        "Selected K": (
            p[
                "K"
            ]
        ),
        "Representative alpha": (
            p[
                "representative_alpha"
            ]
        ),
        "Ensemble members": (
            member_text
        ),
        "Selection reason": (
            p[
                "selection_reason"
            ]
        ),
        "Member-weighted reference full-domain objective": (
            p[
                "full_domain_objective"
            ]
        ),
        "Full-domain valid gridpoint number": (
            p[
                "full_domain_valid_gridpoint_number"
            ]
        ),
    })


paper_df = pd.DataFrame(
    paper_rows
)

paper_df.to_csv(
    TABLE_S4_CSV,
    index=False,
    encoding="utf-8-sig",
)

with pd.ExcelWriter(
    PARAM_XLSX,
    engine="openpyxl",
) as writer:

    all_k_search_df.to_excel(
        writer,
        sheet_name="all_K_alpha_search",
        index=False,
    )

    method_k_summary_df.to_excel(
        writer,
        sheet_name="method_K_autoplateau",
        index=False,
    )

    method_selected_df.to_excel(
        writer,
        sheet_name="method_selected",
        index=False,
    )

    full_reeval_df.to_excel(
        writer,
        sheet_name="full_domain_reevaluation",
        index=False,
    )

    selected_df.to_excel(
        writer,
        sheet_name="final_selection",
        index=False,
    )

    search_points_df.to_excel(
        writer,
        sheet_name="300_search_points",
        index=False,
    )

    split_df.to_excel(
        writer,
        sheet_name="split_information",
        index=False,
    )

    paper_df.to_excel(
        writer,
        sheet_name="TableS4_values",
        index=False,
    )


print(
    "Saved:",
    PARAM_XLSX,
)

print(
    "Saved:",
    TABLE_S4_CSV,
)

print(
    "\n每种optimizer的K自动平台选择："
)
display(
    method_selected_df
)

print(
    "\n最终ensemble/单方法选择："
)
display(
    selected_df
)

print(
    "\n最终Table S4参数："
)
display(
    paper_df
)


In [ ]:
# Cell 9. NetCDF IMF输出工具（压缩等级9）
# 支持single_method与weighted_ensemble两种最终配置


def copy_nc_attrs(
    src_var,
    dst_var,
    exclude=("_FillValue",),
):
    for attr in src_var.ncattrs():
        if attr in exclude:
            continue

        try:
            dst_var.setncattr(
                attr,
                src_var.getncattr(
                    attr
                ),
            )
        except Exception:
            pass


def copy_coordinate(
    dst,
    src_nc,
    coord_name,
    values=None,
):
    src_var = src_nc.variables[
        coord_name
    ]

    kwargs = {}

    if "_FillValue" in src_var.ncattrs():
        kwargs[
            "fill_value"
        ] = src_var.getncattr(
            "_FillValue"
        )

    out = dst.createVariable(
        coord_name,
        src_var.dtype,
        src_var.dimensions,
        **kwargs,
    )

    copy_nc_attrs(
        src_var,
        out,
    )

    if values is None:
        out[:] = src_var[:]
    else:
        out[:] = values

    return out


def build_output_paths(
    output_stem,
    K,
    split_name,
):
    return [
        DATA_DIR
        / f"{output_stem}_imf_{k}_{split_name}.nc"
        for k in range(
            1,
            int(K) + 1,
        )
    ]


def final_selection_signature(
    selected_info,
):
    """
    用最终成员(alpha/K/weight/optimizer)生成可复现签名。
    """
    members = []

    for m in selected_info[
        "members"
    ]:
        members.append({
            "optimizer": str(
                m["optimizer"]
            ),
            "alpha": round(
                float(
                    m["alpha"]
                ),
                10,
            ),
            "K": int(
                m["K"]
            ),
            "weight": round(
                float(
                    m.get(
                        "weight",
                        1.0,
                    )
                ),
                12,
            ),
        })

    payload = {
        "selection_mode": str(
            selected_info[
                "selection_mode"
            ]
        ),
        "K": int(
            selected_info[
                "K"
            ]
        ),
        "members": members,
        "solver_signature": str(
            FAST_SOLVER_SIGNATURE
        ),
    }

    return json.dumps(
        payload,
        sort_keys=True,
        ensure_ascii=False,
        separators=(",", ":"),
    )


def output_files_are_complete(
    paths,
    selected_info,
    split_name,
):
    if not all(
        Path(
            p
        ).exists()
        for p in paths
    ):
        return False

    expected_K = int(
        selected_info[
            "K"
        ]
    )

    expected_signature = (
        final_selection_signature(
            selected_info
        )
    )

    for p in paths:
        try:
            with Dataset(
                p,
                "r",
            ) as nc:

                if int(
                    getattr(
                        nc,
                        "vmd_write_complete",
                        0,
                    )
                ) != 1:
                    return False

                if int(
                    getattr(
                        nc,
                        "vmd_K",
                        -1,
                    )
                ) != expected_K:
                    return False

                if str(
                    getattr(
                        nc,
                        "split",
                        "",
                    )
                ) != split_name:
                    return False

                if str(
                    getattr(
                        nc,
                        "vmd_selection_signature",
                        "",
                    )
                ) != expected_signature:
                    return False

        except Exception:
            return False

    return True


def create_new_imf_files(
    src_nc,
    meta,
    spec,
    split_name,
    time_slice,
    selected_info,
):
    K = int(
        selected_info[
            "K"
        ]
    )

    paths = build_output_paths(
        spec[
            "output_stem"
        ],
        K,
        split_name,
    )

    if any(
        p.exists()
        for p in paths
    ):
        if not OVERWRITE_OUTPUTS:
            raise FileExistsError(
                f"{spec['output_stem']} {split_name}: "
                "发现已有但与当前最终选择不完全匹配的输出文件。"
                "若确认要重算，请设置OVERWRITE_OUTPUTS=True。"
            )

        for p in paths:
            if p.exists():
                p.unlink()

    time_dim = meta[
        "time_dim"
    ]
    lat_dim = meta[
        "lat_dim"
    ]
    lon_dim = meta[
        "lon_dim"
    ]

    ntime = (
        time_slice.stop
        - time_slice.start
    )

    ny = len(
        src_nc.dimensions[
            lat_dim
        ]
    )
    nx = len(
        src_nc.dimensions[
            lon_dim
        ]
    )

    time_values = src_nc.variables[
        time_dim
    ][
        time_slice.start:
        time_slice.stop
    ]

    source_var = meta[
        "var"
    ]

    source_units = getattr(
        source_var,
        "units",
        "",
    )

    selection_signature = (
        final_selection_signature(
            selected_info
        )
    )

    member_details = [
        {
            "optimizer": str(
                m["optimizer"]
            ),
            "alpha": float(
                m["alpha"]
            ),
            "K": int(
                m["K"]
            ),
            "weight": float(
                m.get(
                    "weight",
                    1.0,
                )
            ),
            "full_domain_objective": float(
                m.get(
                    "full_domain_objective",
                    np.nan,
                )
            ),
        }
        for m in selected_info[
            "members"
        ]
    ]

    handles = []
    out_vars = []

    for k, path in enumerate(
        paths,
        start=1,
    ):
        dst = Dataset(
            path,
            "w",
            format="NETCDF4",
        )

        dst.createDimension(
            time_dim,
            ntime,
        )
        dst.createDimension(
            lat_dim,
            ny,
        )
        dst.createDimension(
            lon_dim,
            nx,
        )

        copy_coordinate(
            dst,
            src_nc,
            time_dim,
            values=time_values,
        )

        copy_coordinate(
            dst,
            src_nc,
            lat_dim,
        )

        copy_coordinate(
            dst,
            src_nc,
            lon_dim,
        )

        dst.title = (
            f"VMD IMF {k} of "
            f"{spec['output_stem']} "
            f"for {split_name} split"
        )

        dst.source_file = spec[
            "filename"
        ]
        dst.source_variable = spec[
            "var_name"
        ]
        dst.split = split_name

        dst.vmd_selection_mode = str(
            selected_info[
                "selection_mode"
            ]
        )
        dst.vmd_selected_optimizer = str(
            selected_info[
                "optimizer"
            ]
        )
        dst.vmd_K = int(
            selected_info[
                "K"
            ]
        )

        # ensemble时这里只是汇总代表值；
        # 真正使用的每个alpha保存在member JSON中。
        dst.vmd_alpha = float(
            selected_info[
                "representative_alpha"
            ]
        )
        dst.vmd_alpha_is_representative = int(
            selected_info[
                "selection_mode"
            ]
            == "weighted_ensemble"
        )

        dst.vmd_member_count = int(
            selected_info[
                "member_count"
            ]
        )
        dst.vmd_member_details_json = json.dumps(
            member_details,
            ensure_ascii=False,
            separators=(",", ":"),
        )
        dst.vmd_selection_signature = (
            selection_signature
        )
        dst.vmd_selection_reason = str(
            selected_info[
                "selection_reason"
            ]
        )

        dst.vmd_tau = float(
            TAU
        )
        dst.vmd_DC = int(
            DC
        )
        dst.vmd_init = int(
            INIT
        )
        dst.vmd_tolerance = float(
            TOL
        )
        dst.vmd_max_iter = int(
            VMD_MAX_ITER
        )
        dst.vmd_solver_signature = str(
            FAST_SOLVER_SIGNATURE
        )
        dst.vmd_search_gridpoint_number = int(
            SEARCH_GRIDPOINT_NUMBER
        )

        dst.vmd_final_selection = (
            "Complete K=2..9 scan; alpha and K selected by "
            "automatic low-objective/flat-region criterion without "
            "manual percentage threshold; methods with matching K "
            "are combined by inverse-full-domain-objective weighted "
            "IMF averaging, otherwise the best full-domain candidate "
            "is retained."
        )

        dst.vmd_mode_alignment_note = (
            "Weighted averaging is only performed for methods with "
            "identical K. Before averaging, final modes are explicitly "
            "sorted from low to high by their power-weighted center "
            "frequency at each gridpoint, so corresponding IMF indices "
            "are frequency-aligned across ensemble members."
        )

        dst.split_decomposition = (
            "train, validation, and test are decomposed independently"
        )
        dst.history_context_hours = int(
            TIMESTEP - 1
        )
        dst.split_time_overlap_note = (
            "Adjacent split files overlap by 5 predictor hours "
            "to provide complete 6-hour input context for the first "
            "sample of validation/test; each split is nevertheless "
            "decomposed independently."
        )
        dst.odd_length_handling = (
            "repeat final value once if odd, then truncate after VMD"
        )
        dst.original_predictor_retained = 1
        dst.note = (
            "This IMF is an additional first-stage predictor channel; "
            "the original undecomposed predictor is retained."
        )

        dst.vmd_write_complete = 0
        dst.last_completed_tile = -1

        chunk_t = max(
            1,
            min(
                int(
                    TIME_CHUNK
                ),
                ntime,
            ),
        )

        chunk_y = max(
            1,
            min(
                int(
                    TILE_Y
                ),
                ny,
            ),
        )

        chunk_x = max(
            1,
            min(
                int(
                    TILE_X
                ),
                nx,
            ),
        )

        var_name = (
            f"{spec['var_name']}_imf_{k}"
        )

        vout = dst.createVariable(
            var_name,
            np.float32,
            (
                time_dim,
                lat_dim,
                lon_dim,
            ),
            zlib=True,
            complevel=int(
                COMPRESSION_LEVEL
            ),
            shuffle=True,
            chunksizes=(
                chunk_t,
                chunk_y,
                chunk_x,
            ),
            fill_value=np.float32(
                np.nan
            ),
        )

        if selected_info[
            "selection_mode"
        ] == "weighted_ensemble":
            vout.long_name = (
                f"Weighted-ensemble VMD intrinsic mode function {k} "
                f"of {spec['output_stem']}"
            )
        else:
            vout.long_name = (
                f"VMD intrinsic mode function {k} "
                f"of {spec['output_stem']}"
            )

        if source_units:
            vout.units = (
                source_units
            )

        vout.imf_index = int(
            k
        )

        handles.append(
            dst
        )
        out_vars.append(
            vout
        )

    return (
        handles,
        out_vars,
        paths,
        0,
    )


def close_all(
    handles,
):
    if handles is None:
        return

    for h in handles:
        try:
            h.close()
        except Exception:
            pass


In [ ]:
# Cell 10. 最终train/valid/test全域VMD
#
# single_method:
#   直接输出该方法VMD结果。
#
# weighted_ensemble:
#   对具有相同K的2/3个方法分别执行VMD，
#   按Cell 7给出的full-domain objective反比权重逐IMF加权平均后输出。


def _cpu_decompose_prepared_to_modes(
    prepared,
    alpha,
    K,
    ntime,
):
    fp, nit = _vmd_positive_batch_numba(
        prepared["f_hat_pos"],
        prepared["freqs_pos"],
        float(alpha),
        int(K),
        float(TOL),
        int(VMD_MAX_ITER),
        int(prepared["T_full"]),
    )

    G = int(prepared["n_series"])
    P = int(fp.shape[1])
    K = int(K)

    # 与GPU最终分解一致：按最终功率加权中心频率从低到高排序，
    # 确保不同alpha/optimizer的同编号IMF可以安全对应。
    centers = np.empty((G, K), dtype=np.float64)
    freqs = np.asarray(prepared["freqs_pos"], dtype=np.float64)

    for kk in range(K):
        pos = fp[:, :, kk]
        power = (
            pos.real.astype(np.float64) ** 2
            + pos.imag.astype(np.float64) ** 2
        )
        denom = np.maximum(np.sum(power, axis=1), 1e-30)
        centers[:, kk] = np.sum(
            power * freqs[None, :],
            axis=1,
        ) / denom
        del power

    mode_order = np.argsort(centers, axis=1)

    modes = np.empty(
        (
            G,
            K,
            int(ntime),
        ),
        dtype=np.float32,
    )

    for target_kk in range(K):
        source_kk = mode_order[:, target_kk]
        gather_idx = np.broadcast_to(
            source_kk[:, None, None],
            (G, P, 1),
        )
        pos_sorted = np.take_along_axis(
            fp,
            gather_idx,
            axis=2,
        )[:, :, 0]

        modes[
            :,
            target_kk,
            :,
        ] = reconstruct_mode_matrix_from_positive(
            pos_sorted,
            original_len=int(
                ntime
            ),
            T_full=int(
                prepared[
                    "T_full"
                ]
            ),
            fft_workers=FFT_WORKERS,
        )
        del pos_sorted

    del fp, nit, centers, mode_order

    return modes


def decompose_variable_split_to_nc(
    src_path,
    spec,
    split_name,
    time_slice,
    selected_info,
):
    K = int(
        selected_info[
            "K"
        ]
    )

    members = [
        dict(
            m
        )
        for m in selected_info[
            "members"
        ]
    ]

    if len(
        members
    ) < 1:
        raise ValueError(
            f"{spec['output_stem']}: 最终选择没有成员。"
        )

    for m in members:
        if int(
            m[
                "K"
            ]
        ) != K:
            raise ValueError(
                f"{spec['output_stem']}: ensemble成员K不一致，"
                "不同K禁止逐IMF加权平均。"
            )

    weights = np.asarray(
        [
            float(
                m.get(
                    "weight",
                    1.0,
                )
            )
            for m in members
        ],
        dtype=np.float64,
    )

    if (
        np.any(
            ~np.isfinite(
                weights
            )
        )
        or np.any(
            weights < 0
        )
        or np.sum(
            weights
        ) <= 0
    ):
        raise ValueError(
            f"{spec['output_stem']}: 非法ensemble权重。"
        )

    weights = weights / np.sum(
        weights
    )

    for m, w in zip(
        members,
        weights,
    ):
        m[
            "weight"
        ] = float(
            w
        )

    paths = build_output_paths(
        spec[
            "output_stem"
        ],
        K,
        split_name,
    )

    if (
        SKIP_COMPLETE_SPLIT
        and output_files_are_complete(
            paths,
            selected_info=selected_info,
            split_name=split_name,
        )
    ):
        print(
            f"[SKIP COMPLETE] "
            f"{spec['output_stem']} "
            f"{split_name}"
        )
        return paths

    print(
        "\n"
        + "=" * 110
    )

    print(
        f"[FINAL WHOLE-DOMAIN VMD] "
        f"{spec['output_stem']} | "
        f"split={split_name} | "
        f"mode={selected_info['selection_mode']} | "
        f"K={K} | "
        f"members={len(members)}"
    )

    for m in members:
        print(
            f"  {m['optimizer']}: "
            f"alpha={float(m['alpha']):.8f}, "
            f"K={int(m['K'])}, "
            f"weight={float(m['weight']):.8f}, "
            f"Jfull={float(m.get('full_domain_objective', np.nan)):.8f}"
        )

    print(
        "=" * 110
    )

    with Dataset(
        src_path,
        "r",
    ) as src_nc:

        meta = detect_dimensions(
            src_nc,
            spec[
                "var_name"
            ],
        )

        src_var = meta[
            "var"
        ]

        ny = len(
            src_nc.dimensions[
                meta[
                    "lat_dim"
                ]
            ]
        )

        nx = len(
            src_nc.dimensions[
                meta[
                    "lon_dim"
                ]
            ]
        )

        ntime = int(
            time_slice.stop
            - time_slice.start
        )

        cube = read_tyx_fast(
            src_var,
            meta,
            time_slice=time_slice,
            y_slice=slice(
                0,
                ny,
            ),
            x_slice=slice(
                0,
                nx,
            ),
        )

        matrix_all = np.ascontiguousarray(
            np.transpose(
                cube,
                (
                    1,
                    2,
                    0,
                ),
            ).reshape(
                ny * nx,
                ntime,
            ),
            dtype=np.float32,
        )

        del cube

        valid_mask = raw_series_matrix_valid_mask(
            matrix_all
        )

        valid_idx = np.flatnonzero(
            valid_mask
        )

        if valid_idx.size == 0:
            raise RuntimeError(
                f"{spec['output_stem']} "
                f"{split_name}: 无有效格点。"
            )

        valid_matrix = np.ascontiguousarray(
            matrix_all[
                valid_idx
            ]
        )

        del matrix_all
        gc.collect()

        print(
            f"raw valid grids="
            f"{valid_idx.size}/{ny*nx}, "
            f"matrix="
            f"{valid_matrix.nbytes/1024**3:.2f} GiB"
        )

        prepared = prepare_vmd_spectra_matrix(
            valid_matrix,
            fft_workers=FFT_WORKERS,
        )

        del valid_matrix
        gc.collect()

        pbar = tqdm(
            total=int(
                valid_idx.size
            )
            * len(
                members
            ),
            desc=(
                f"{spec['output_stem']} "
                f"{split_name} "
                f"{selected_info['selection_mode']}"
            ),
            unit="grid-method",
            dynamic_ncols=True,
        )

        modes_valid = None

        try:
            if COMPUTE_DEVICE.startswith(
                "cuda"
            ):
                gpu_cache = (
                    prepared_to_torch_cache(
                        prepared
                    )
                )

                for im, m in enumerate(
                    members
                ):
                    # gpu函数会重置pbar total，因此这里传None，
                    # 外层统一按成员×格点更新。
                    member_modes = (
                        gpu_decompose_single_candidate_to_modes(
                            gpu_cache,
                            alpha=float(
                                m[
                                    "alpha"
                                ]
                            ),
                            K=K,
                            pbar=None,
                            desc=(
                                f"{spec['output_stem']} "
                                f"{split_name} "
                                f"{m['optimizer']}"
                            ),
                        )
                    )

                    w = np.float32(
                        m[
                            "weight"
                        ]
                    )

                    member_modes *= w

                    if modes_valid is None:
                        # 第一成员直接转交所有权，避免再复制一份巨型IMF数组。
                        modes_valid = member_modes
                        member_modes = None
                    else:
                        modes_valid += member_modes
                        del member_modes

                    pbar.update(
                        int(
                            valid_idx.size
                        )
                    )
                    pbar.set_postfix_str(
                        (
                            f"{m['optimizer']} "
                            f"a={float(m['alpha']):.4f} "
                            f"w={float(m['weight']):.4f}"
                        ),
                        refresh=True,
                    )

                    gc.collect()

                free_torch_cache(
                    gpu_cache
                )

            else:
                for m in members:
                    member_modes = (
                        _cpu_decompose_prepared_to_modes(
                            prepared,
                            alpha=float(
                                m[
                                    "alpha"
                                ]
                            ),
                            K=K,
                            ntime=ntime,
                        )
                    )

                    w = np.float32(
                        m[
                            "weight"
                        ]
                    )

                    member_modes *= w

                    if modes_valid is None:
                        # 第一成员直接转交所有权，避免再复制一份巨型IMF数组。
                        modes_valid = member_modes
                        member_modes = None
                    else:
                        modes_valid += member_modes
                        del member_modes

                    pbar.update(
                        int(
                            valid_idx.size
                        )
                    )

                    pbar.set_postfix_str(
                        (
                            f"{m['optimizer']} "
                            f"a={float(m['alpha']):.4f} "
                            f"w={float(m['weight']):.4f}"
                        ),
                        refresh=True,
                    )

                    gc.collect()

        finally:
            pbar.close()

        if modes_valid is None:
            raise RuntimeError(
                "最终VMD未生成任何IMF。"
            )

        del prepared
        gc.collect()

        # 创建/覆盖输出文件
        if any(
            p.exists()
            for p in paths
        ):
            if not OVERWRITE_OUTPUTS:
                raise FileExistsError(
                    f"{spec['output_stem']} {split_name}: "
                    "存在旧的非完整或与当前selection signature不匹配的输出。"
                    "确认需要覆盖后设置OVERWRITE_OUTPUTS=True。"
                )

            for p in paths:
                if p.exists():
                    p.unlink()

        (
            handles,
            out_vars,
            paths,
            _,
        ) = create_new_imf_files(
            src_nc,
            meta,
            spec,
            split_name,
            time_slice,
            selected_info,
        )

        try:
            for kk in tqdm(
                range(
                    K
                ),
                desc=(
                    f"{spec['output_stem']} "
                    f"{split_name} write IMF"
                ),
                dynamic_ncols=True,
            ):
                full_mode = np.full(
                    (
                        ny * nx,
                        ntime,
                    ),
                    np.nan,
                    dtype=np.float32,
                )

                full_mode[
                    valid_idx
                ] = modes_valid[
                    :,
                    kk,
                    :,
                ]

                full_mode = (
                    full_mode.reshape(
                        ny,
                        nx,
                        ntime,
                    ).transpose(
                        2,
                        0,
                        1,
                    )
                )

                out_vars[
                    kk
                ][
                    :,
                    :,
                    :,
                ] = full_mode

                handles[
                    kk
                ].sync()

                del full_mode

            for h in handles:
                h.setncattr(
                    "last_completed_tile",
                    1,
                )
                h.setncattr(
                    "vmd_write_complete",
                    1,
                )
                h.setncattr(
                    "whole_domain_matrix",
                    1,
                )
                h.setncattr(
                    "compute_backend",
                    COMPUTE_DEVICE,
                )
                h.sync()

        finally:
            close_all(
                handles
            )

        print(
            "host final IMF cache peak "
            f"(valid only) ~= "
            f"{modes_valid.nbytes/1024**3:.2f} GiB"
        )

        del (
            modes_valid,
            valid_idx,
            valid_mask,
        )
        gc.collect()

    return paths


all_output_paths = []

for variable_key, spec in SOURCE_SPECS.items():
    selected_info = selected_params[
        variable_key
    ]

    src_path = SOURCE_DIR / spec[
        "filename"
    ]

    for split_name in [
        "train",
        "valid",
        "test",
    ]:
        paths = decompose_variable_split_to_nc(
            src_path=src_path,
            spec=spec,
            split_name=split_name,
            time_slice=predictor_slices[
                split_name
            ],
            selected_info=selected_info,
        )

        all_output_paths.extend(
            paths
        )

        gc.collect()


print(
    "\n全部split-specific VMD分解完成。"
)


In [ ]:
# Cell 11. 输出完整性、压缩、样本数和重构QC

verification = []

for variable_key, spec in SOURCE_SPECS.items():
    p = selected_params[variable_key]
    K = int(p["K"])

    for split_name in [
        "train",
        "valid",
        "test",
    ]:
        paths = build_output_paths(
            spec["output_stem"],
            K,
            split_name,
        )

        expected_time = (
            predictor_slices[split_name].stop
            - predictor_slices[split_name].start
        )

        expected_samples = (
            sample_slices[split_name].stop
            - sample_slices[split_name].start
        )

        for k, path in enumerate(
            paths,
            start=1,
        ):
            if not path.exists():
                raise FileNotFoundError(path)

            var_name = (
                f"{spec['var_name']}_imf_{k}"
            )

            with Dataset(path, "r") as nc:
                v = nc.variables[var_name]

                filters = (
                    v.filters()
                    if hasattr(v, "filters")
                    else {}
                )

                verification.append({
                    "variable": variable_key,
                    "split": split_name,
                    "imf": k,
                    "filename": path.name,
                    "time_length": int(v.shape[0]),
                    "expected_time_length": int(
                        expected_time
                    ),
                    "window_recovered_samples": int(
                        v.shape[0]
                        - TIMESTEP
                        + 1
                    ),
                    "expected_samples": int(
                        expected_samples
                    ),
                    "compression_level": filters.get(
                        "complevel",
                        None,
                    ),
                    "write_complete": int(
                        getattr(
                            nc,
                            "vmd_write_complete",
                            0,
                        )
                    ),
                })

verification_df = pd.DataFrame(
    verification
)

bad = verification_df[
    (verification_df["time_length"]
     != verification_df["expected_time_length"])
    |
    (
        verification_df[
            "window_recovered_samples"
        ]
        != verification_df[
            "expected_samples"
        ]
    )
    |
    (
        verification_df[
            "compression_level"
        ]
        != COMPRESSION_LEVEL
    )
    |
    (
        verification_df[
            "write_complete"
        ]
        != 1
    )
]

if len(bad) > 0:
    print("发现输出检查异常：")
    display(bad)
    raise RuntimeError(
        "VMD输出完整性检查未通过。"
    )

print("全部NetCDF完整性检查通过。")
print(
    verification_df.groupby(
        ["variable", "split"]
    ).size()
)


# 重构QC：每变量/每split检查中心格点
qc_records = []

for variable_key, spec in SOURCE_SPECS.items():
    src_path = SOURCE_DIR / spec["filename"]
    p = selected_params[variable_key]
    K = int(p["K"])

    with Dataset(src_path, "r") as src_nc:
        meta = detect_dimensions(
            src_nc,
            spec["var_name"],
        )

        ny = len(
            src_nc.dimensions[
                meta["lat_dim"]
            ]
        )
        nx = len(
            src_nc.dimensions[
                meta["lon_dim"]
            ]
        )

        iy = ny // 2
        ix = nx // 2

        for split_name in [
            "train",
            "valid",
            "test",
        ]:
            slc = predictor_slices[
                split_name
            ]

            original = read_point_series(
                meta["var"],
                meta,
                slc,
                iy,
                ix,
            )

            reconstructed = np.zeros_like(
                original,
                dtype=np.float64,
            )

            for k in range(1, K + 1):
                path = (
                    DATA_DIR
                    / f"{spec['output_stem']}_imf_{k}_{split_name}.nc"
                )

                var_name = (
                    f"{spec['var_name']}_imf_{k}"
                )

                with Dataset(path, "r") as nc:
                    reconstructed += np.asarray(
                        nc.variables[var_name][
                            :, iy, ix
                        ],
                        dtype=np.float64,
                    )

            rmse = float(
                np.sqrt(
                    np.nanmean(
                        (
                            reconstructed
                            - original
                        ) ** 2
                    )
                )
            )

            denom = float(
                np.nanstd(original)
            )

            nrmse = (
                rmse / denom
                if denom > 0
                else np.nan
            )

            qc_records.append({
                "variable": variable_key,
                "split": split_name,
                "iy": iy,
                "ix": ix,
                "reconstruction_RMSE": rmse,
                "reconstruction_RMSE_over_std": nrmse,
            })

qc_df = pd.DataFrame(qc_records)

print("\n中心格点VMD重构QC：")
display(qc_df)

print("\n最终Table S4参数：")
display(paper_df)

print("\n输出文件示例：")
for p in all_output_paths[:15]:
    print(p.name)
